# for colab


## §0. Setup


### 0.1 安裝依賴


In [ ]:
!pip uninstall -y unsloth unsloth_zoo

# Core HF stack
!pip install -q -U --no-cache-dir \
  "transformers==4.56.2" \
  "peft==0.18.0" \
  "trl==0.19.1" \
  "accelerate>=1.4.0,<2.0.0" \
  "bitsandbytes>=0.45.0" \
  "hf_transfer"

# Install matching latest Unsloth + Unsloth Zoo together
!pip install -q -U --no-cache-dir --force-reinstall --no-deps \
  "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

!pip install -q -U --no-cache-dir --force-reinstall --no-deps \
  "unsloth @ git+https://github.com/unslothai/unsloth.git"

# RAG / parse
!pip install -q pymupdf sentence-transformers FlagEmbedding rank_bm25 faiss-cpu nltk

# Utils
!pip install -q matplotlib seaborn tqdm scikit-learn pandas

!pip check


In [ ]:
# ═════════════════════════════════════════════════════════════
# v9: 全部 import 集中在這個 cell（下游 cell 不再 import）
# ═════════════════════════════════════════════════════════════

from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from transformers import EarlyStoppingCallback
from torch.utils.data import WeightedRandomSampler


# ─── stdlib ───
import os, re, json, csv, time, gc, math, random, warnings, shutil
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Optional

# ─── 數據 / 視覺化 ───
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# ─── PyTorch ───
import torch
import torch.nn as nn
from torch.utils.data import WeightedRandomSampler

# ─── HuggingFace 生態 ───
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel,
)
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

# ─── PDF parse / RAG ───
import fitz  # PyMuPDF
import nltk
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss

# ─── Sklearn / API ───
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from openai import OpenAI

# ─── Colab ───
from google.colab import drive

# ─── Unsloth (last; touches CUDA at import) ───
from unsloth import FastLanguageModel

# ─── nltk one-time setup ───
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings("ignore")
print("✅ All v9 imports loaded")


### 0.2 Mount Google Drive + Local Cache（v4 fix issue D）
資料路徑：`MyDrive/GAI/dataset/`

**Cache 策略**：parse / retrieval cache 寫**本機 SSD** `/content/GAI/cache`（Drive 對 1000+ 小檔案有 10× 寫入延遲）；每個 split 完成後再整包 sync 到 Drive 持久化。新 session restore：Drive → 本機。

- `CACHE_DIR` (本機 SSD) — 既有程式碼指向這裡
- `DRIVE_CACHE_DIR` — source of truth；§1.3.5 / §2.4.5 sync 用
- `CKPT_DIR / OUTPUTS_DIR` 保持 Drive（少量大檔，I/O 沒問題）


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DATA_DIR = "/content/drive/MyDrive/GAI/dataset"
LOCAL_CACHE_DIR = "/content/GAI/cache"                            # 本機 SSD
DRIVE_CACHE_DIR = "/content/drive/MyDrive/GAI/cache"              # Drive，sync 用
CACHE_DIR = LOCAL_CACHE_DIR                                       # alias，既有程式碼指向本機
CKPT_DIR = "/content/drive/MyDrive/GAI/checkpoints"
OUTPUTS_DIR = "/content/drive/MyDrive/GAI/outputs"

PARSED_CACHE_NAME = "parsed_pdfs_v0"                                 # e.g. "parsed_pdfs_v1"
RETRIEVED_CACHE_NAME = "retrieved_v0"                            # your Drive folder name; change to "retrieved_v2" if needed
PARSED_CACHE_DIR = f"{CACHE_DIR}/{PARSED_CACHE_NAME}"
RETRIEVED_CACHE_DIR = f"{CACHE_DIR}/{RETRIEVED_CACHE_NAME}"
DRIVE_PARSED_CACHE_DIR = f"{DRIVE_CACHE_DIR}/{PARSED_CACHE_NAME}"
DRIVE_RETRIEVED_CACHE_DIR = f"{DRIVE_CACHE_DIR}/{RETRIEVED_CACHE_NAME}"

# 建本機 + Drive 子目錄
for base in [LOCAL_CACHE_DIR, DRIVE_CACHE_DIR]:
    for sub in [PARSED_CACHE_NAME, f"{RETRIEVED_CACHE_NAME}/train", f"{RETRIEVED_CACHE_NAME}/dev", f"{RETRIEVED_CACHE_NAME}/test"]:
        os.makedirs(f"{base}/{sub}", exist_ok=True)
for d in [CKPT_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Restore：if Drive 有 cache 而本機沒有，cp 回本機（避免重 parse / retrieval）
def _restore(sub: str):
    src = f"{DRIVE_CACHE_DIR}/{sub}"
    dst = f"{LOCAL_CACHE_DIR}/{sub}"
    n_drive = len(os.listdir(src)) if os.path.exists(src) else 0
    n_local = len(os.listdir(dst)) if os.path.exists(dst) else 0
    if n_drive > 0 and n_local == 0:
        print(f"  ⏬ Restoring {sub}: Drive→Local ({n_drive} files)...")
        os.makedirs(dst, exist_ok=True)
        # cp -r 比 shutil.copytree 快
        os.system(f'cp -r "{src}/." "{dst}/"')
        print(f"     done → local now has {len(os.listdir(dst))} files")
    elif n_drive > 0:
        print(f"  ✓ {sub}: local already has {n_local} files (Drive has {n_drive})")
    else:
        print(f"  · {sub}: no cache yet (Drive={n_drive}, local={n_local})")

print("Cache restore from Drive (if any):")
for sub in [PARSED_CACHE_NAME, f"{RETRIEVED_CACHE_NAME}/train", f"{RETRIEVED_CACHE_NAME}/dev", f"{RETRIEVED_CACHE_NAME}/test"]:
    _restore(sub)

print("\nActive cache profile:")
print(f"  PARSED_CACHE_DIR    = {PARSED_CACHE_DIR}")
print(f"  RETRIEVED_CACHE_DIR = {RETRIEVED_CACHE_DIR}")

# 確認 dataset
print("\nDataset:")
for f in ["train.csv", "dev.csv", "test.csv", "classes.json"]:
    path = f"{DATA_DIR}/{f}"
    print(f"  {'✓' if os.path.exists(path) else '✗'} {path}")

for split in ["train", "dev", "test"]:
    pdf_dir = f"{DATA_DIR}/paper_evidence/{split}"
    n = len(os.listdir(pdf_dir)) if os.path.exists(pdf_dir) else 0
    print(f"  {split}: {n} PDFs")


### 0.3 Imports & Config
所有超參數集中在這個 cell，方便調整。


In [ ]:
import os
import re
import json
import csv
import time
import gc
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ───── 5-class definitions (從 classes.json 讀) ─────
with open(f"{DATA_DIR}/classes.json", encoding="utf-8") as f:
    CLASSES = json.load(f)
ID2CONCEPT = {c["id"]: c["concept"] for c in CLASSES}
CONCEPT2ID = {c["concept"]: c["id"] for c in CLASSES}
CONCEPT_LOWER2ID = {c["concept"].lower(): c["id"] for c in CLASSES}
print(f"5 classes: {ID2CONCEPT}")

# ───── Chunking (constants — 切換值移到 §0.6/§0.7) ─────
CHUNK_SENT_OVERLAP = 2

# ───── Retrieval (internals — 切換值移到 §0.6/§0.7) ─────
DENSE_TOP_K = 60
BM25_TOP_K = 60
RRF_K = 60
RRF_TOP_K = 30
RERANK_POOL = 30
ALWAYS_INCLUDE_SECTIONS = set()  # 結構偏置

# ───── Training (constants — LoRA r/alpha/lr/epochs/modules 移到 §0.6/§0.7) ─────
USE_UNSLOTH = True
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 10000
LORA_DROPOUT = 0.05
PER_DEVICE_BATCH = 1  # OOM at 2: logits buffer 1.65 GiB
GRAD_ACCUM = 16       # effective batch = 16
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.5
EVAL_STEPS = 30
SAVE_STEPS = 30

# ───── Inference ─────
GEN_MAX_NEW_TOKENS = 12

print("Config loaded (v9: hyper-params 切換值由 §0.6 master config + §0.7 resolver 提供)")


### 0.4 GPU 檢查與設定
確認 A100 / bf16 / FA2 可用。


In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        FP16, BF16 = False, True
    else:
        DTYPE = torch.float16
        FP16, BF16 = True, False
        # V100 fp16 fallback
        LR = 1e-4
        MAX_SEQ_LENGTH = 2048
        PER_DEVICE_BATCH = 2
        GRAD_ACCUM = 8
        MAX_GRAD_NORM = 0.3
        USE_UNSLOTH = False
        print("→ V100 fp16 fallback：lr=1e-4, batch=2x8, max_seq=2048, max_grad_norm=0.3, no Unsloth")
else:
    raise RuntimeError("no GPU detected")


### 0.5 Dataset 概覽 (VIZ)
5 類分布長條圖、text 長度統計。


In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
dev_df = pd.read_csv(f"{DATA_DIR}/dev.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"Train: {len(train_df):,} ({train_df['paper_id'].nunique()} papers)")
print(f"Dev:   {len(dev_df):,} ({dev_df['paper_id'].nunique()} papers)")
print(f"Test:  {len(test_df):,} ({test_df['paper_id'].nunique()} papers)")
print(f"Paper overlap: train∩dev={len(set(train_df['paper_id']) & set(dev_df['paper_id']))}, "
      f"train∩test={len(set(train_df['paper_id']) & set(test_df['paper_id']))}, "
      f"dev∩test={len(set(dev_df['paper_id']) & set(test_df['paper_id']))}")

# 5 類分布
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, name in [(axes[0], train_df, "Train"), (axes[1], dev_df, "Dev")]:
    counts = df['label'].value_counts().sort_index()
    labels = [f"{i}: {ID2CONCEPT[i]}" for i in counts.index]
    sns.barplot(x=labels, y=counts.values, ax=ax, palette="viridis")
    ax.set_title(f"{name} ({len(df):,}) — Max:Min = {counts.max() / counts.min():.1f}:1")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=30)
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()

# Text 長度
text_lens = train_df['text'].str.len()
print(f"\nReview text length: min={text_lens.min()} median={int(text_lens.median())} "
      f"mean={int(text_lens.mean())} p95={int(text_lens.quantile(0.95))} max={text_lens.max()}")


### 0.6 ★ v9 MASTER CONFIG — 改這個 cell 切變體
所有 v9 五軸（R retrieval / H LoRA / I aug+loss+inference / P prompt / E ensemble）的 switch 集中於此。
下面所有 cell 透過 §0.7 resolver 讀對應 constants，**不需要每次改 build_notebook.py**。

**Default = T3 winner config**（重現 dev 0.55 / public 0.69）。要試新變體只改這個 cell。


In [ ]:
# ───── 軸 R: Retrieval ─────
RETRIEVAL_VARIANT = "R0"

# ───── 軸 H: LoRA / Optimizer ─────
LORA_VARIANT = "H1"


# ───── 軸 I: Augmentation + Loss + Inference ─────
AUG_VARIANT = "T3"

LOSS_VARIANT = "ce_w"

USE_SAMPLER = True       # ★ T3 winner（WeightedRandomSampler，固定不調）

INFERENCE_VARIANT = "greedy"

# ───── 軸 P: Prompt ─────
PROMPT_VARIANT = "cot_lite"

RETRIEVAL_PRESETS = {
    "R0":   dict(target=6800, max=9000, merge_min=3000, topk=5, max_evidence=10000, embedding="bge-large"),
}

LORA_PRESETS = {
    "H1":   dict(r=64,  alpha=128, lr=5e-5, epochs=2, modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]),
}
AUG_PRESETS = {
    "T3": dict(num=0,   temp=0,   method="none"),
}
LOSS_PRESETS = {
    "ce_w": dict(type="weighted_ce", beta=0.999),
}
INFERENCE_PRESETS = {
    "greedy":          dict(k=1, temps=[0.0]),
}

# ───── 軸 E: Ensemble ─────
ENSEMBLE_SEEDS = [42, 9222, 786349]

# ───── Warm-start ─────
RESUME_FROM_CHECKPOINT = None



### 0.7 Variant Resolver — preset → constants
把 §0.6 的 switch 翻譯成下游 cell 直接讀的 constants（CHUNK_TARGET_CHARS, LORA_R, LR, NUM_EPOCHS, ...）。
**不需要修改 §0.7 來切變體**，只改 §0.6 即可。


In [ ]:


# ───── Resolve to downstream constants ─────
_R = RETRIEVAL_PRESETS[RETRIEVAL_VARIANT]
CHUNK_TARGET_CHARS = _R["target"]
CHUNK_MAX_CHARS    = _R["max"]
MERGE_MIN_CHARS    = _R["merge_min"]
EVIDENCE_TOP_K     = _R["topk"]
MAX_EVIDENCE_CHARS = _R["max_evidence"]
EMBEDDING_NAME     = _R["embedding"]

_H = LORA_PRESETS[LORA_VARIANT]
LORA_R              = _H["r"]
LORA_ALPHA          = _H["alpha"]
LR                  = _H["lr"]
NUM_EPOCHS          = _H["epochs"]
LORA_TARGET_MODULES = _H["modules"]

_A = AUG_PRESETS[AUG_VARIANT]
NUMBER_AUG_TARGET   = _A["num"]
TEMPORAL_AUG_TARGET = _A["temp"]
AUG_METHOD          = _A["method"]

_L = LOSS_PRESETS[LOSS_VARIANT]
LOSS_TYPE           = _L["type"]
EFF_NUM_BETA        = _L.get("beta", 0.999)
LDAM_MARGIN_C       = _L.get("margin_C", 0.5)
LDAM_SCALE          = _L.get("s", 30)
USE_CLASS_WEIGHT    = (LOSS_TYPE == "weighted_ce")  # downstream §5.A reads this

_I = INFERENCE_PRESETS[INFERENCE_VARIANT]
TTA_K     = _I["k"]
TTA_TEMPS = _I["temps"]

# ───── Local + Drive checkpoint paths（v9 ckpt 策略：本地寫，最終 sync） ─────
LOCAL_CKPT_DIR = "/content/GAI/checkpoints"   # 本地 SSD，訓練期間每 save_steps 寫此處
DRIVE_CKPT_DIR = CKPT_DIR                     # 既有 §0.2 已定義為 /content/drive/MyDrive/GAI/checkpoints
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)



## §1. PDF Parsing


### 1.1 Parser 函數
PyMuPDF 抽文字 + section 偵測 + aggressive strip（references/appendix/figure caption）。

**重要**：原始 PDF 平均 7.6 MB（含 figures），需要強力 strip 否則 token 爆炸污染檢索。


In [ ]:
# ── Canonical section keyword → category；parser 只切 major section，subsection title 會被跳過
_SECTION_CATEGORIES = {
    "abstract": "abstract",
    "introduction": "introduction",
    "related work": "related_work", "related works": "related_work",
    "background": "related_work", "prior work": "related_work",
    "literature": "related_work", "literature review": "related_work",
    "preliminaries": "related_work", "preliminary": "related_work",
    "method": "methodology", "methods": "methodology",
    "methodology": "methodology", "approach": "methodology",
    "model": "methodology", "framework": "methodology",
    "system": "methodology", "proposed": "methodology",
    "proposed method": "methodology", "proposed approach": "methodology",
    "architecture": "methodology", "network": "methodology", "networks": "methodology",
    "experiment": "experiments", "experiments": "experiments",
    "experimental": "experiments", "experimental setup": "experiments",
    "setup": "experiments", "evaluation": "experiments",
    "implementation": "experiments", "implementation details": "experiments",
    "result": "results", "results": "results",
    "analysis": "results", "discussion": "results",
    "ablation": "results", "ablations": "results",
    "ablation study": "results", "finding": "results",
    "findings": "results", "performance": "results",
    "conclusion": "conclusion", "conclusions": "conclusion",
    "summary": "conclusion", "future work": "conclusion",
    "data": "data", "dataset": "data", "datasets": "data",
    "corpus": "data", "corpora": "data",
}

_SKIP_HEADERS = {
    "acknowledgment", "acknowledgments", "acknowledgement", "acknowledgements",
    "references", "reference", "bibliography",
    "appendix", "appendices", "supplementary",
    "supplementary material", "supplementary materials",
    "ethics statement", "broader impact", "broader impacts",
    "limitations", "limitation",
    # NeurIPS / ICLR-style appendix checklists
    "paper checklist", "neurips paper checklist", "checklist",
    "reproducibility checklist", "datasheet",
}

_NUM_HEAD_PAT = re.compile(r"^\s*(\d+(?:\.\d+){0,3})\.?\s+(.+?)\s*$")
_LETTER_HEAD_PAT = re.compile(r"^\s*([A-Z](?:\.\d+)*)\.?\s+(.+?)\s*$")
_PAGE_NUM_PAT = re.compile(r"^\s*\d+\s*$")
_FIG_TABLE_PAT = re.compile(r"^\s*(?:Figure|Fig\.|Table|Algorithm|Listing)\s*S?\d+", re.IGNORECASE)
_YEAR_PAT = re.compile(r"\b(?:19|20)\d{2}\b")

_TITLE_STOPWORDS = {
    "a", "an", "the", "of", "and", "or", "in", "for", "to", "with",
    "on", "at", "by", "from", "as", "via", "vs", "is", "are", "be",
    "no", "not", "but", "into", "over", "under",
}

_SENTENCE_STARTERS = {
    "while", "if", "when", "where", "although", "since", "because",
    "however", "moreover", "furthermore", "thus", "therefore",
    "given", "let", "consider", "first", "next", "finally", "once",
    "even", "despite", "though", "after", "before", "during",
    "in", "on", "at", "by", "for", "with", "to", "from", "as",
    "this", "these", "those", "it", "we", "our", "they",
}


def _strip_numbering(header: str) -> str:
    h = re.sub(r"^\s*(?:\d+(?:\.\d+){0,3}|[A-Z](?:\.\d+)*)\.?\s+", "", header).strip()
    return re.sub(r"\s+", " ", h)


def _is_checklist_header(header: str) -> bool:
    return "checklist" in _strip_numbering(header).lower()


def _classify_section(header: str, parent_category: str = "content") -> str:
    h = _strip_numbering(header).lower()
    if not h:
        return parent_category if parent_category != "content" else "content"
    if h in _SKIP_HEADERS:
        return "_skip"
    if h in _SECTION_CATEGORIES:
        return _SECTION_CATEGORIES[h]
    for kw, cat in _SECTION_CATEGORIES.items():
        if re.search(rf"\b{re.escape(kw)}\b", h):
            return cat
    for kw in _SKIP_HEADERS:
        if h.startswith(kw):
            return "_skip"
    return parent_category if parent_category != "content" else "content"


def _is_title_case(text: str) -> bool:
    tokens = text.split()
    n_cap, n_violation = 0, 0
    for tok in tokens:
        clean = re.sub(r"^[^A-Za-z]+", "", tok)
        if not clean: continue
        if clean[0].isupper():
            n_cap += 1
        elif clean.lower() in _TITLE_STOPWORDS:
            continue
        else:
            n_violation += 1
    return n_cap >= 1 and n_violation == 0


def _looks_like_header_text(text: str, short_len: int = 60) -> bool:
    s = text.strip()
    if not s or not s[0].isalpha() or not s[0].isupper():
        return False
    first_word = s.split()[0].lower().rstrip(",.;:")
    if first_word in _SENTENCE_STARTERS:
        return False
    if len(s) <= short_len:
        if "," in s and any(w in s.lower().split() for w in ["which", "that", "where", "when"]):
            return False
        return True
    return _is_title_case(s)


def _has_real_word(text: str, min_len: int = 4) -> bool:
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", text)
    return any(len(w) >= min_len and w.lower() not in _TITLE_STOPWORDS for w in words)


def _looks_like_bib_entry(text: str) -> bool:
    if "(cited on page" in text.lower():
        return True
    if _YEAR_PAT.search(text) and text.count(",") >= 2:
        return True
    return False


def _estimate_body_font_size(font_sizes: list[float]) -> float:
    '''Estimate prose body font. PDFs with many tables can have tiny table text as the raw mode.'''
    rounded = [round(float(s), 1) for s in font_sizes if s and float(s) > 0]
    prose_like = [s for s in rounded if 8.5 <= s <= 11.5]
    if prose_like:
        return Counter(prose_like).most_common(1)[0][0]
    return Counter(rounded).most_common(1)[0][0] if rounded else 10.0


# v10: unnumbered major headings are common in these PDFs
# e.g. "Introduction", "Background", "Related Work", "Experiments".
_UNNUMBERED_OK_HEADERS = {
    "abstract",
}

_CORE_UNNUMBERED_MAJOR_HEADERS = {
    "abstract", "introduction",
    "background", "preliminaries", "preliminary",
    "related work", "related works", "prior work",
    "method", "methods", "methodology",
    "data", "dataset", "datasets",
    "experiment", "experiments", "evaluation",
    "result", "results", "discussion",
    "conclusion", "conclusions", "references",
    "acknowledgment", "acknowledgments", "acknowledgement", "acknowledgements",
}

def _has_heading_style(font_size: float, body_size: float, is_bold: bool) -> bool:
    return bool(is_bold) or (body_size > 0 and font_size >= body_size * 1.08)


def _has_major_heading_style(font_size: float, body_size: float, is_bold: bool) -> bool:
    return bool(is_bold) and body_size > 0 and font_size >= body_size * 1.15


def _has_split_heading_title_style(font_size: float, body_size: float, is_bold: bool) -> bool:
    # Some papers render the standalone section number at 12pt but the title line at ~10.5pt bold.
    # The number line itself must still satisfy _has_major_heading_style before this is used.
    return bool(is_bold) and body_size > 0 and font_size >= body_size * 1.02


def _is_unnumbered_major_header(text: str, font_size: float, body_size: float, is_bold: bool) -> bool:
    h_low = text.lower().rstrip(".").strip()
    if h_low in _UNNUMBERED_OK_HEADERS:
        return True
    if not _has_heading_style(font_size, body_size, is_bold):
        return False
    if h_low in _CORE_UNNUMBERED_MAJOR_HEADERS:
        return _has_major_heading_style(font_size, body_size, is_bold)
    if h_low in _SECTION_CATEGORIES or h_low in _SKIP_HEADERS:
        return _has_major_heading_style(font_size, body_size, is_bold)
    cls = _classify_section(text)
    return cls != "content" and _has_major_heading_style(font_size, body_size, is_bold) and _looks_like_header_text(text, short_len=90)


def _merge_split_numbered_headings(all_lines: list[tuple[str, float, bool]], body_size: float) -> list[tuple[str, float, bool]]:
    '''Some PDFs extract "3" / "A" and the title as two separate lines.
    Merge only when both lines look like major heading typography, so normal page/table numbers stay untouched.
    '''
    merged = []
    i = 0
    while i < len(all_lines):
        text, size, bold = all_lines[i]
        t = text.strip().rstrip(".")
        if (
            (
                (re.fullmatch(r"\d{1,2}", t) and 1 <= int(t) <= 30)
                or re.fullmatch(r"[A-Z]", t)
            )
            and _has_major_heading_style(size, body_size, bold)
            and i + 1 < len(all_lines)
        ):
            next_text, next_size, next_bold = all_lines[i + 1]
            nt = next_text.strip()
            if (
                nt
                and len(nt) <= 120
                and not nt.isdigit()
                and not _NUM_HEAD_PAT.match(nt)
                and _has_real_word(nt)
                and _has_split_heading_title_style(next_size, body_size, next_bold)
                and not _looks_like_bib_entry(nt)
            ):
                merged.append((f"{t}. {nt}", max(size, next_size), bool(bold or next_bold)))
                i += 2
                continue
        merged.append((text, size, bold))
        i += 1
    return merged

def _detect_header_kind(text: str, font_size: float, body_size: float, is_bold: bool):
    '''v9 大標題 only — 回傳 4 種：
       'major'      → 整數編號 "N. Title" 或 unnumbered_ok（Abstract）→ 開新 section
       'subsection' → dotted 編號 "3.1 ..."（只跳過標題；正文留在目前 major section）
       'sibling'    → 無編號但 match canonical keyword（Method/Model/Implementation Details/...）
                      → 只跳過標題，不開新 section，也不塞回 parent body
       'skip'       → References / Appendix / Acknowledgments / Bibliography
       None         → body line
    '''
    t = text.strip()
    if not t or len(t) > 120: return None
    if _FIG_TABLE_PAT.match(t): return None
    if t.endswith((",", ";", "!", "?")): return None
    if t.endswith(".") and len(t) > 60: return None
    if len(t) <= 2 or t.isdigit(): return None
    if any(ch in t for ch in "@∗†‡§"): return None
    if _looks_like_bib_entry(t): return None
    if re.search(r"[=<>≤≥≈≠]", t) or t.count(":") >= 2: return None

    if re.fullmatch(r"[A-Z]\.\d+(?:\.\d+)*\.?", t):
        return "subsection"

    m = _NUM_HEAD_PAT.match(t)
    if m:
        num_str, title = m.group(1), m.group(2).strip()
        if not title:
            return None
        if "." in num_str:
            # dotted (3.1 / 3.1.2) → subsection 候選
            if _has_real_word(title):
                return "subsection"
            return None
        try:
            main_num = int(num_str)
        except ValueError:
            return None
        if not (1 <= main_num <= 30): return None
        if not _has_real_word(title):
            return None
        title_cls = _classify_section(title)
        if re.search(r"\d", title) and title_cls == "content":
            return None
        if not (_is_title_case(title) or _looks_like_header_text(title, short_len=90)):
            return None
        return "major"

    lm = _LETTER_HEAD_PAT.match(t)
    if lm:
        letter_num, title = lm.group(1), lm.group(2).strip()
        if "." in letter_num:
            return "subsection" if _has_real_word(title) else None
        if not _has_major_heading_style(font_size, body_size, is_bold):
            return None
        if not _has_real_word(title):
            return None
        return "major"

    h_low = t.lower().rstrip(".").strip()
    if _is_unnumbered_major_header(t, font_size, body_size, is_bold):
        return "major"
    return None


def parse_pdf(pdf_path: str) -> dict:
    '''字型 + 編號 pattern + keyword 三層 header 偵測；保留全文，只用 major headings 切 section。'''
    doc = fitz.open(pdf_path)
    n_pages = doc.page_count

    all_lines = []  # (text, font_size, is_bold)
    font_sizes = []
    for i in range(n_pages):
        page = doc[i]
        d = page.get_text("dict")
        for block in d.get("blocks", []):
            if block.get("type", 0) != 0: continue
            for line in block.get("lines", []):
                spans = line.get("spans", []) or []
                if not spans: continue
                texts, sizes, bold_n = [], [], 0
                for sp in spans:
                    texts.append(sp.get("text", ""))
                    sizes.append(float(sp.get("size", 0)))
                    if int(sp.get("flags", 0)) & 16:
                        bold_n += 1
                line_text = "".join(texts).strip()
                if not line_text: continue
                avg_size = sum(sizes) / len(sizes) if sizes else 0.0
                is_bold = bold_n >= max(1, len(spans) // 2)
                all_lines.append((line_text, avg_size, is_bold))
                font_sizes.append(round(avg_size, 1))
    doc.close()

    if not all_lines:
        return {"paper_id": Path(pdf_path).stem, "num_pages": n_pages,
                "sections": [], "total_chars": 0}

    body_size = _estimate_body_font_size(font_sizes)
    all_lines = _merge_split_numbered_headings(all_lines, body_size)

    sections, current = [], None
    skip_mode = False
    pre_abstract_mode = True
    last_section_num = 0
    checklist_mode = False

    for text, size, bold in all_lines:
        if _PAGE_NUM_PAT.match(text) and len(text) <= 4: continue

        if checklist_mode and current is not None:
            current["body"] = (current["body"] + " " + text).strip() if current["body"] else text
            continue

        kind = _detect_header_kind(text, size, body_size, bold)

        if kind == "skip":
            kind = "major"

        if kind == "subsection":
            # dotted (3.1 / 3.1.1) → 不開 section，也不把 subsection title 塞進 parent body
            continue

        if kind == "sibling":
            # 無編號 keyword：
            #   - 其他 → 視為 sub-heading，只跳過標題，不另開 section、不併入 parent body
            continue

        if kind == "major":
            cls = _classify_section(text)
            if cls == "_skip":
                # References / Appendix / checklist-like headers are kept as content.
                cls = "content"
            m = _NUM_HEAD_PAT.match(text.strip())
            if m and "." not in m.group(1):
                try:
                    hn = int(m.group(1))
                except ValueError:
                    hn = None
                if hn is not None:
                    last_section_num = hn
            if pre_abstract_mode:
                if cls == "abstract" or (m and "." not in m.group(1)):
                    pre_abstract_mode = False
                else:
                    continue
            skip_mode = False
            checklist_mode = _is_checklist_header(text)
            current = {"category": cls, "header": text.strip(), "body": "", "level": 1}
            sections.append(current)
            continue

        # body line
        if pre_abstract_mode or skip_mode or current is None: continue
        current["body"] = (current["body"] + " " + text).strip() if current["body"] else text

    # Fallback：完全沒抓到 abstract（罕見）— 用同一套邏輯，但允許從第一個 major 起跑
    if not sections:
        current = None; skip_mode = False; last_section_num = 0
        checklist_mode = False
        for text, size, bold in all_lines:
            if _PAGE_NUM_PAT.match(text) and len(text) <= 4: continue
            if checklist_mode and current is not None:
                current["body"] = (current["body"] + " " + text).strip() if current["body"] else text
                continue
            kind = _detect_header_kind(text, size, body_size, bold)
            if kind == "skip":
                kind = "major"
            if kind == "subsection":
                continue
            if kind == "sibling":
                continue
            if kind == "major":
                cls = _classify_section(text)
                if cls == "_skip":
                    cls = "content"
                m = _NUM_HEAD_PAT.match(text.strip())
                if m and "." not in m.group(1):
                    try: hn = int(m.group(1))
                    except ValueError: hn = None
                    if hn is not None: last_section_num = hn
                skip_mode = False
                checklist_mode = _is_checklist_header(text)
                current = {"category": cls, "header": text.strip(), "body": "", "level": 1}
                sections.append(current)
                continue
            if skip_mode or current is None: continue
            current["body"] = (current["body"] + " " + text).strip() if current["body"] else text

    sections = [s for s in sections if s["body"].strip() and len(s["body"]) >= 60]

    return {
        "paper_id": Path(pdf_path).stem,
        "num_pages": n_pages,
        "sections": sections,
        "total_chars": sum(len(s["body"]) for s in sections),
    }

print("Parser ready (font-aware section detection).")


In [ ]:
# 切小標題


# # ── Canonical section keyword → category；subsection 透過 parent 繼承
# _SECTION_CATEGORIES = {
#     "abstract": "abstract",
#     "introduction": "introduction",
#     "related work": "related_work", "related works": "related_work",
#     "background": "related_work", "prior work": "related_work",
#     "literature": "related_work", "literature review": "related_work",
#     "preliminaries": "related_work", "preliminary": "related_work",
#     "method": "methodology", "methods": "methodology",
#     "methodology": "methodology", "approach": "methodology",
#     "model": "methodology", "framework": "methodology",
#     "system": "methodology", "proposed": "methodology",
#     "proposed method": "methodology", "proposed approach": "methodology",
#     "architecture": "methodology",
#     "experiment": "experiments", "experiments": "experiments",
#     "experimental": "experiments", "experimental setup": "experiments",
#     "setup": "experiments", "evaluation": "experiments",
#     "implementation": "experiments", "implementation details": "experiments",
#     "result": "results", "results": "results",
#     "analysis": "results", "discussion": "results",
#     "ablation": "results", "ablations": "results",
#     "ablation study": "results", "finding": "results",
#     "findings": "results", "performance": "results",
#     "conclusion": "conclusion", "conclusions": "conclusion",
#     "summary": "conclusion", "future work": "conclusion",
#     "data": "data", "dataset": "data", "datasets": "data",
#     "corpus": "data", "corpora": "data",
# }

# _SKIP_HEADERS = {
#     "acknowledgment", "acknowledgments", "acknowledgement", "acknowledgements",
#     "references", "reference", "bibliography",
#     "appendix", "appendices", "supplementary",
#     "supplementary material", "supplementary materials",
#     "ethics statement", "broader impact", "broader impacts",
#     "limitations", "limitation",
# }

# _NUM_HEAD_PAT = re.compile(r"^\s*(\d+(?:\.\d+){0,3})\.?\s+(.+?)\s*$")
# _PAGE_NUM_PAT = re.compile(r"^\s*\d+\s*$")
# _FIG_TABLE_PAT = re.compile(r"^\s*(?:Figure|Fig\.|Table|Algorithm|Listing)\s*\d+", re.IGNORECASE)
# _YEAR_PAT = re.compile(r"\b(?:19|20)\d{2}\b")

# _TITLE_STOPWORDS = {
#     "a", "an", "the", "of", "and", "or", "in", "for", "to", "with",
#     "on", "at", "by", "from", "as", "via", "vs", "is", "are", "be",
#     "no", "not", "but", "into", "over", "under",
# }

# _SENTENCE_STARTERS = {
#     "while", "if", "when", "where", "although", "since", "because",
#     "however", "moreover", "furthermore", "thus", "therefore",
#     "given", "let", "consider", "first", "next", "finally", "once",
#     "even", "despite", "though", "after", "before", "during",
#     "in", "on", "at", "by", "for", "with", "to", "from", "as",
#     "this", "these", "those", "it", "we", "our", "they",
# }


# def _strip_numbering(header: str) -> str:
#     h = re.sub(r"^\s*\d+(?:\.\d+){0,3}\.?\s*", "", header).strip()
#     return re.sub(r"\s+", " ", h)


# def _classify_section(header: str, parent_category: str = "content") -> str:
#     h = _strip_numbering(header).lower()
#     if not h:
#         return parent_category if parent_category != "content" else "content"
#     if h in _SKIP_HEADERS:
#         return "_skip"
#     if h in _SECTION_CATEGORIES:
#         return _SECTION_CATEGORIES[h]
#     for kw, cat in _SECTION_CATEGORIES.items():
#         if re.search(rf"\b{re.escape(kw)}\b", h):
#             return cat
#     for kw in _SKIP_HEADERS:
#         if h.startswith(kw):
#             return "_skip"
#     return parent_category if parent_category != "content" else "content"


# def _is_title_case(text: str) -> bool:
#     tokens = text.split()
#     n_cap, n_violation = 0, 0
#     for tok in tokens:
#         clean = re.sub(r"^[^A-Za-z]+", "", tok)
#         if not clean: continue
#         if clean[0].isupper():
#             n_cap += 1
#         elif clean.lower() in _TITLE_STOPWORDS:
#             continue
#         else:
#             n_violation += 1
#     return n_cap >= 1 and n_violation == 0


# def _looks_like_header_text(text: str, short_len: int = 60) -> bool:
#     s = text.strip()
#     if not s or not s[0].isalpha() or not s[0].isupper():
#         return False
#     first_word = s.split()[0].lower().rstrip(",.;:")
#     if first_word in _SENTENCE_STARTERS:
#         return False
#     if len(s) <= short_len:
#         if "," in s and any(w in s.lower().split() for w in ["which", "that", "where", "when"]):
#             return False
#         return True
#     return _is_title_case(s)


# def _has_real_word(text: str, min_len: int = 4) -> bool:
#     words = re.findall(r"[A-Za-z][A-Za-z'-]*", text)
#     return any(len(w) >= min_len and w.lower() not in _TITLE_STOPWORDS for w in words)


# def _looks_like_bib_entry(text: str) -> bool:
#     if "(cited on page" in text.lower():
#         return True
#     if _YEAR_PAT.search(text) and text.count(",") >= 2:
#         return True
#     return False


# def _is_likely_header(text: str, font_size: float, body_size: float, is_bold: bool):
#     t = text.strip()
#     if not t or len(t) > 120: return False, 0
#     if t.endswith((",", ";", "!", "?")) and len(t) > 30: return False, 0
#     if t.endswith(".") and len(t) > 60: return False, 0
#     if len(t) <= 2 or t.isdigit(): return False, 0
#     if any(ch in t for ch in "@∗†‡§"): return False, 0
#     if _looks_like_bib_entry(t): return False, 0
#     if re.search(r"[=<>≤≥≈≠]", t) or t.count(":") >= 2: return False, 0

#     # A. keyword exact
#     h_low = _strip_numbering(t).lower()
#     if h_low in _SECTION_CATEGORIES or h_low in _SKIP_HEADERS:
#         m = _NUM_HEAD_PAT.match(t)
#         if m:
#             return True, m.group(1).count(".") + 1
#         return True, 1

#     # B. numbered + Title Case + 真詞 + 標題不含數字
#     m = _NUM_HEAD_PAT.match(t)
#     if m:
#         num_str, title = m.group(1), m.group(2).strip()
#         try:
#             main_num = int(num_str.split(".")[0])
#         except ValueError:
#             main_num = 0
#         if (1 <= main_num <= 30 and title
#                 and not re.search(r"\d", title)
#                 and _is_title_case(title) and _has_real_word(title)):
#             return True, num_str.count(".") + 1
#         return False, 0

#     # C/D：>= 2 字 + header-like + 真詞 + 至多 1 個數字 token
#     if len(t.split()) < 2 or not _looks_like_header_text(t) or not _has_real_word(t):
#         return False, 0
#     if len(re.findall(r"\b\d+(?:\.\d+)*\b", t)) > 1:
#         return False, 0

#     # C. larger font
#     if body_size > 0 and font_size >= body_size * 1.10 and len(t) < 100:
#         return True, 1
#     # D. bold + short
#     if is_bold and len(t) < 80 and not t.endswith("."):
#         return True, 1
#     return False, 0


# def parse_pdf(pdf_path: str) -> dict:
#     '''字型 + 編號 pattern + keyword 三層 header 偵測；剝除 references/appendix/figure caption。'''
#     doc = fitz.open(pdf_path)
#     n_pages = doc.page_count

#     all_lines = []  # (text, font_size, is_bold)
#     font_sizes = []
#     for i in range(n_pages):
#         page = doc[i]
#         d = page.get_text("dict")
#         for block in d.get("blocks", []):
#             if block.get("type", 0) != 0: continue
#             for line in block.get("lines", []):
#                 spans = line.get("spans", []) or []
#                 if not spans: continue
#                 texts, sizes, bold_n = [], [], 0
#                 for sp in spans:
#                     texts.append(sp.get("text", ""))
#                     sizes.append(float(sp.get("size", 0)))
#                     if int(sp.get("flags", 0)) & 16:
#                         bold_n += 1
#                 line_text = "".join(texts).strip()
#                 if not line_text: continue
#                 avg_size = sum(sizes) / len(sizes) if sizes else 0.0
#                 is_bold = bold_n >= max(1, len(spans) // 2)
#                 all_lines.append((line_text, avg_size, is_bold))
#                 font_sizes.append(round(avg_size, 1))
#     doc.close()

#     if not all_lines:
#         return {"paper_id": Path(pdf_path).stem, "num_pages": n_pages,
#                 "sections": [], "total_chars": 0}

#     body_size = Counter(font_sizes).most_common(1)[0][0]

#     sections, current = [], None
#     skip_mode = False
#     pre_abstract_mode = True
#     parent_cat = "content"

#     for text, size, bold in all_lines:
#         if _PAGE_NUM_PAT.match(text) and len(text) <= 4: continue
#         if _FIG_TABLE_PAT.match(text) and len(text) < 120: continue

#         is_hdr, level = _is_likely_header(text, size, body_size, bold)
#         if is_hdr:
#             cls = _classify_section(text, parent_cat if level >= 2 else "content")
#             if cls == "_skip":
#                 skip_mode = True; current = None; continue
#             if pre_abstract_mode:
#                 if cls == "abstract":
#                     pre_abstract_mode = False
#                 else:
#                     continue  # 標題/作者/affiliation 區塊，跳過
#             skip_mode = False
#             if level <= 1 and cls != "content":
#                 parent_cat = cls
#             current = {"category": cls, "header": text.strip(), "body": "", "level": level}
#             sections.append(current)
#         else:
#             if pre_abstract_mode or skip_mode or current is None: continue
#             current["body"] = (current["body"] + " " + text).strip() if current["body"] else text

#     # Fallback：完全沒抓到 abstract（罕見）
#     if not sections:
#         current = None; skip_mode = False
#         for text, size, bold in all_lines:
#             if _PAGE_NUM_PAT.match(text) and len(text) <= 4: continue
#             if _FIG_TABLE_PAT.match(text) and len(text) < 120: continue
#             is_hdr, level = _is_likely_header(text, size, body_size, bold)
#             if is_hdr:
#                 cls = _classify_section(text)
#                 if cls == "_skip":
#                     skip_mode = True; current = None; continue
#                 skip_mode = False
#                 current = {"category": cls, "header": text.strip(), "body": "", "level": level}
#                 sections.append(current)
#             else:
#                 if skip_mode or current is None: continue
#                 current["body"] = (current["body"] + " " + text).strip() if current["body"] else text

#     sections = [s for s in sections if s["body"].strip() and len(s["body"]) >= 60]

#     return {
#         "paper_id": Path(pdf_path).stem,
#         "num_pages": n_pages,
#         "sections": sections,
#         "total_chars": sum(len(s["body"]) for s in sections),
#     }

# print("Parser ready (font-aware section detection).")


### 1.2 [QC] 在 3 篇 PDF 上手檢 parser
目視 section 切分品質：應該看到 Abstract / Introduction / numbered sections (1, 2, 3, 3.1, 3.2 ...) 都有切出來。


In [ ]:
# shutil.rmtree(f"{LOCAL_CACHE_DIR}/parsed_pdfs", ignore_errors=True);
# shutil.rmtree(f"{DRIVE_CACHE_DIR}/parsed_pdfs", ignore_errors=True)

In [ ]:
# sample_pdfs = sorted(Path(f"{DATA_DIR}/paper_evidence/train").glob("*.pdf"))[:5]
# for pdf in sample_pdfs:
#     print(f"\n📄 {pdf.name} ({pdf.stat().st_size / 1024:.1f} KB)")
#     r = parse_pdf(str(pdf))
#     print(f"   pages={r['num_pages']}, sections={len(r['sections'])}, total_chars={r['total_chars']:,}")
#     print(f"   sections (level, category, header, body_len):")
#     for s in r['sections'][:20]:
#         lvl = s.get('level', 0)
#         hdr = (s['header'][:55]) if s['header'] else "<no header>"
#         print(f"     L{lvl} [{s['category']:13s}] {hdr:55s} ({len(s['body']):,} chars)")
#     if len(r['sections']) > 20:
#         print(f"     ... +{len(r['sections']) - 20} more sections")


### 1.3 全量 parse → cache 到 Drive
**一次性執行**（942 + 120 + 120 = 1,182 篇 PDF，約 30–60 分鐘）。

如果 cache 已存在會跳過該檔，所以 session 中斷可重跑無風險。


In [ ]:
def parse_all(split: str) -> tuple[int, int]:
    pdf_dir = Path(f"{DATA_DIR}/paper_evidence/{split}")
    out_dir = Path(f"{CACHE_DIR}/parsed_pdfs")
    out_dir.mkdir(parents=True, exist_ok=True)
    pdfs = sorted(pdf_dir.glob("*.pdf"))
    success, failed = 0, []
    for pdf in tqdm(pdfs, desc=f"Parse {split}"):
        out_path = out_dir / f"{pdf.stem}.json"
        if out_path.exists():
            success += 1; continue
        try:
            r = parse_pdf(str(pdf))
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(r, f, ensure_ascii=False)
            success += 1
        except Exception as e:
            failed.append((pdf.name, str(e)))
    if failed:
        print(f"  ❌ {len(failed)} failed: {failed[:5]}")
    return success, len(failed)

for split in ["train", "dev", "test"]:
    s, f = parse_all(split)
    print(f"{split}: {s} success, {f} failed")


### 1.3.5 Sync parsed cache → Drive（v4 fix issue D）
跑完 §1.3 後執行：把本機 `LOCAL_CACHE_DIR/parsed_pdfs` 整包 sync 到 Drive 持久化。
session 結束（12hr 上限或斷線）前**一定要跑這個 cell**，不然 `/content` 清空就會失去 cache。


In [ ]:
import time
src = f"{LOCAL_CACHE_DIR}/parsed_pdfs"
dst = f"{DRIVE_CACHE_DIR}/parsed_pdfs"
n_local = len(os.listdir(src)) if os.path.exists(src) else 0
print(f"Local parsed cache: {n_local} files")
if n_local == 0:
    print("⚠️  本機沒有 parsed cache，跳過 sync")
else:
    os.makedirs(dst, exist_ok=True)
    t0 = time.time()
    rc = os.system(f'cp -r "{src}/." "{dst}/"')
    elapsed = time.time() - t0
    n_drive = len(os.listdir(dst))
    total_mb = sum(os.path.getsize(f'{dst}/{f}') for f in os.listdir(dst)) / 1024 / 1024
    print(f"✅ Synced {n_drive} files ({total_mb:.1f} MB) Local→Drive in {elapsed:.1f}s")


### 1.4 [VIZ] Parsed 統計


In [ ]:
parsed_dir = Path(f"{CACHE_DIR}/parsed_pdfs")
all_parsed = list(parsed_dir.glob("*.json"))
print(f"Total parsed: {len(all_parsed)}")

stats = []
section_rows = []
for p in tqdm(all_parsed, desc="Loading parsed"):
    with open(p, encoding="utf-8") as f:
        d = json.load(f)
    sections = d.get("sections", [])
    stats.append({
        "paper_id": d["paper_id"],
        "n_sections": len(sections),
        "total_chars": d.get("total_chars", sum(len(s.get("body", "")) for s in sections)),
        "categories": Counter(s.get("category", "unknown") for s in sections),
    })
    for idx, s in enumerate(sections):
        body = s.get("body", "")
        section_rows.append({
            "paper_id": d["paper_id"],
            "section_idx": idx,
            "category": s.get("category", "unknown"),
            "header": s.get("header", ""),
            "body_chars": len(body),
        })

stats_df = pd.DataFrame(stats)
section_len_df = pd.DataFrame(section_rows)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(stats_df['total_chars'], bins=40)
axes[0].set_title("Parsed text length (chars)")
axes[0].set_xlabel("chars"); axes[0].set_ylabel("# papers")
axes[1].hist(stats_df['n_sections'], bins=20)
axes[1].set_title("Sections per paper")
axes[1].set_xlabel("# sections")
axes[2].hist(section_len_df['body_chars'], bins=60)
axes[2].set_title("Section body length (chars)")
axes[2].set_xlabel("chars"); axes[2].set_ylabel("# sections")
plt.tight_layout(); plt.show()

print(f"\nText chars: median={int(stats_df['total_chars'].median()):,}, "
      f"mean={int(stats_df['total_chars'].mean()):,}, "
      f"p95={int(stats_df['total_chars'].quantile(0.95)):,}")

print(f"\nSections: total={len(section_len_df):,}, "
      f"per-paper median={stats_df['n_sections'].median():.1f}, "
      f"per-paper mean={stats_df['n_sections'].mean():.1f}, "
      f"per-paper p95={stats_df['n_sections'].quantile(0.95):.1f}")

q = section_len_df['body_chars'].quantile([0, .25, .5, .75, .9, .95, .99, 1.0]).astype(int)
print("\nSection body chars quantiles:")
for k, v in q.items():
    print(f"  p{int(k*100):>3}: {v:,}")


## §2. Evidence Retrieval (RAG)


### 2.1 Chunking + 共用 helpers
Paragraph-level（300–500 tokens）+ 2-sentence overlap，保留 section metadata。


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def split_sentences(text: str) -> list[str]:
    raw = nltk.sent_tokenize(text)
    out, buf = [], ""
    for s in raw:
        s = s.strip()
        if not s: continue
        if buf:
            buf += " " + s
            if len(buf) >= 100:
                out.append(buf); buf = ""
        elif len(s) < 100:
            buf = s
        else:
            out.append(s)
    if buf:
        if out: out[-1] += " " + buf
        else: out.append(buf)
    return out

@dataclass
class Chunk:
    chunk_id: int
    text: str
    section: str

def merge_small_chunks(chunks: list[Chunk],
                        min_size: int = 400, max_merge: int = 1800) -> list[Chunk]:
    '''v4 fix issue C: 相鄰 chunk 同 category 且前者 < min_size → 合併（不超過 max_merge）。
    保留 category metadata；short fragments 不會單獨進 retrieval（BM25 信號弱）。'''
    out: list[Chunk] = []
    for ch in chunks:
        if (out and out[-1].section == ch.section and len(out[-1].text) < min_size
                and len(out[-1].text) + 1 + len(ch.text) <= max_merge):
            out[-1] = Chunk(out[-1].chunk_id, out[-1].text + " " + ch.text, out[-1].section)
            continue
        out.append(ch)
    for i, c in enumerate(out):
        c.chunk_id = i
    return out

def build_chunks(parsed: dict) -> list[Chunk]:
    chunks: list[Chunk] = []
    for sec in parsed["sections"]:
        sentences = split_sentences(sec["body"])
        if not sentences: continue
        cur, start = [], 0
        for i, sent in enumerate(sentences):
            cur_text = " ".join(cur + [sent])
            if cur and len(cur_text) > CHUNK_TARGET_CHARS:
                chunks.append(Chunk(len(chunks), " ".join(cur), sec["category"]))
                ov = max(0, len(cur) - CHUNK_SENT_OVERLAP)
                cur = cur[ov:] + [sent]
                start = i - len(cur) + 1
            else:
                cur.append(sent)
        if cur:
            chunks.append(Chunk(len(chunks), " ".join(cur), sec["category"]))
    # Force-split oversized
    out = []
    for ch in chunks:
        if len(ch.text) <= CHUNK_MAX_CHARS:
            out.append(ch)
        else:
            t = ch.text; mid = len(t) // 2
            sp = t.rfind(". ", mid - 100, mid + 100)
            sp = sp if sp != -1 else mid
            out.append(Chunk(len(out), t[:sp + 1].strip(), ch.section))
            out.append(Chunk(len(out), t[sp + 1:].strip(), ch.section))
    for i, ch in enumerate(out):
        ch.chunk_id = i
    # v4 fix issue C: merge 小 chunks（v9: min_size 從 §0.7 resolver 讀；max_merge 隨 chunk_max 放寬）
    _max_merge = max(CHUNK_MAX_CHARS, MERGE_MIN_CHARS * 2)
    out = merge_small_chunks(out, min_size=MERGE_MIN_CHARS, max_merge=_max_merge)
    return out

def load_parsed(paper_id: str) -> Optional[dict]:
    path = Path(f"{CACHE_DIR}/parsed_pdfs/{paper_id}.json")
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

print("Chunker ready (with merge_small_chunks post-process, v4 fix issue C).")


### 2.A Embedding Model（v9 dispatcher：依 §0.6 `RETRIEVAL_VARIANT.embedding` 自動載入）
- `bge-large` (default, R_T3/R0/R1/R2/R3) — 335M, MTEB 64.2, 速度中
- `e5-mistral` (R4 variant) — 7B, MTEB 66.6, A100 40GB 才跑 + 慢 10x

不用手動執行多 cell；改 §0.6 `RETRIEVAL_VARIANT="R4"` 就會載 e5-mistral。


In [ ]:
from sentence_transformers import SentenceTransformer

if EMBEDDING_NAME == "bge-large":
    EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"
    EMBED_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
    EMBED_PASSAGE_PREFIX = ""
    EMBED_BATCH = 32
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
elif EMBEDDING_NAME == "e5-mistral":
    EMBED_MODEL_NAME = "intfloat/e5-mistral-7b-instruct"
    EMBED_QUERY_PREFIX = "Instruct: Given a peer review sentence, retrieve relevant passages from the paper.\nQuery: "
    EMBED_PASSAGE_PREFIX = ""
    EMBED_BATCH = 8
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda", torch_dtype=torch.float16)
else:
    raise ValueError(f"Unknown EMBEDDING_NAME={EMBEDDING_NAME}")

print(f"✅ Loaded {EMBED_MODEL_NAME} (variant={EMBEDDING_NAME}, batch={EMBED_BATCH})")


### 2.2 Reranker (BAAI/bge-reranker-v2-m3)
所有 embedding variant 共用同一個 reranker。


In [ ]:
RERANKER_NAME = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(RERANKER_NAME, max_length=1024, device="cuda")
print(f"✅ Loaded {RERANKER_NAME}")


### 2.3 Hybrid Retrieval (Dense + BM25 + RRF + Rerank + 結構偏置)
- Dense top 60 + BM25 top 60 → RRF 融合 top 30 → reranker → top 5
- 結構偏置：強制納入 abstract + conclusion 各 1 個 chunk


In [ ]:

def _tok(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

class PaperRetriever:
    def __init__(self, chunks: list[Chunk]):
        self.chunks = chunks
        if not chunks:
            self.index = self.bm25 = None; return
        texts = [(EMBED_PASSAGE_PREFIX + c.text) if EMBED_PASSAGE_PREFIX else c.text for c in chunks]
        embs = embed_model.encode(
            texts, batch_size=EMBED_BATCH, normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")
        self.index = faiss.IndexFlatIP(embs.shape[1])
        self.index.add(embs)
        self.bm25 = BM25Okapi([_tok(t) for t in texts])

    def dense(self, q: str, top_k: int):
        if self.index is None: return []
        prefixed = (EMBED_QUERY_PREFIX + q) if EMBED_QUERY_PREFIX else q
        emb = embed_model.encode([prefixed], normalize_embeddings=True).astype("float32")
        scores, ids = self.index.search(emb, min(top_k, self.index.ntotal))
        return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i >= 0]

    def bm25_search(self, q: str, top_k: int):
        if self.bm25 is None: return []
        scores = self.bm25.get_scores(_tok(q))
        ranked = sorted(range(len(scores)), key=lambda i: -scores[i])[:top_k]
        return [(i, float(scores[i])) for i in ranked]

    @staticmethod
    def rrf_fuse(*rls, k: int = RRF_K):
        sc = defaultdict(float)
        for rl in rls:
            for rank, (cid, _) in enumerate(rl):
                sc[cid] += 1.0 / (k + rank + 1)
        return sorted(sc, key=lambda x: -sc[x])

    def rerank(self, q: str, ids: list[int], top_k: int):
        if not ids: return []
        pairs = [(q, self.chunks[i].text) for i in ids]
        scores = reranker.predict(pairs, show_progress_bar=False)
        ranked = sorted(zip(ids, [float(s) for s in scores]), key=lambda x: -x[1])
        return ranked[:top_k]

    def retrieve(self, query: str, top_k: int = EVIDENCE_TOP_K) -> list[dict]:
        dense_hits = self.dense(query, DENSE_TOP_K)
        bm25_hits = self.bm25_search(query, BM25_TOP_K)
        fused = self.rrf_fuse(dense_hits, bm25_hits)[:RRF_TOP_K]
        reranked = self.rerank(query, fused, top_k=RERANK_POOL)

        # 結構偏置：強制納入 abstract + conclusion 中分數最高的 1 個
        forced = []
        for sec_target in ALWAYS_INCLUDE_SECTIONS:
            cand = [(cid, s) for cid, s in reranked if self.chunks[cid].section == sec_target]
            if cand:
                forced.append(cand[0])
        # Final selection
        seen = set(cid for cid, _ in forced)
        out = list(forced)
        for cid, s in reranked:
            if cid in seen: continue
            out.append((cid, s)); seen.add(cid)
            if len(out) >= top_k: break

        return [{"chunk_id": cid, "score": s, "text": self.chunks[cid].text,
                 "section": self.chunks[cid].section} for cid, s in out[:top_k]]

print("Retriever ready.")


### 2.3.5 [QC] 先跑 3 篇 paper 檢查 retrieval 品質
先不要跑完整 §2.4。這格只抽幾篇 paper 建 index + retrieval，印出 chunk 長度與 top evidence。

如果品質看起來 OK，再跑 §2.4 全量 cache。


In [ ]:
def get_paper_ids(split: str) -> dict:
    df = {"train": train_df, "dev": dev_df, "test": test_df}[split]
    grouped = defaultdict(list)
    for row in df.itertuples():
        grouped[row.paper_id].append({
            "id": row.id,
            "text": row.text,
            "label": getattr(row, "label", None),
        })
    return grouped

def retrieve_preview_papers(split: str = "dev",
                            n_papers: int = 3,
                            samples_per_paper: int = 2,
                            force_rebuild: bool = True,
                            write_cache: bool = True,
                            paper_ids: list[str] | None = None):
    grouped = get_paper_ids(split)
    out_dir = Path(f"{RETRIEVED_CACHE_DIR}/{split}")
    out_dir.mkdir(parents=True, exist_ok=True)

    if paper_ids is None:
        rng = random.Random(SEED)
        paper_ids = rng.sample(sorted(grouped.keys()), min(n_papers, len(grouped)))
    else:
        paper_ids = [p for p in paper_ids if p in grouped][:n_papers]

    print(f"Preview retrieval split={split}, papers={paper_ids}")
    print(f"R={RETRIEVAL_VARIANT} | chunk target/max={CHUNK_TARGET_CHARS}/{CHUNK_MAX_CHARS} | "
          f"topK={EVIDENCE_TOP_K} | max_evidence={MAX_EVIDENCE_CHARS} | embed={EMBEDDING_NAME}")

    preview_results = {}
    for paper_id in paper_ids:
        print("\n" + "=" * 90)
        print(f"Paper: {paper_id}")
        parsed = load_parsed(paper_id)
        if parsed is None:
            print(f"  ⚠ no parsed cache for {paper_id}")
            continue

        chunks = build_chunks(parsed)
        if not chunks:
            print(f"  ⚠ no chunks for {paper_id}")
            continue

        chunk_lens = np.array([len(c.text) for c in chunks])
        print(f"  parsed sections={len(parsed.get('sections', []))}, chunks={len(chunks)}")
        print(f"  chunk chars: min={chunk_lens.min():,}, median={int(np.median(chunk_lens)):,}, "
              f"p90={int(np.percentile(chunk_lens, 90)):,}, max={chunk_lens.max():,}")
        print(f"  chunks by section: {dict(Counter(c.section for c in chunks).most_common())}")

        retr = PaperRetriever(chunks)
        samples = grouped[paper_id]
        results = {}
        for s in tqdm(samples, desc=f"Retrieve preview {paper_id}"):
            evidence = retr.retrieve(s["text"], top_k=EVIDENCE_TOP_K)
            results[s["id"]] = {"text": s["text"], "label": s["label"], "evidence": evidence}

        if write_cache:
            out_path = out_dir / f"{paper_id}.json"
            if force_rebuild or not out_path.exists():
                with open(out_path, "w", encoding="utf-8") as f:
                    json.dump(results, f, ensure_ascii=False)
                print(f"  ✅ wrote preview cache → {out_path}")
            else:
                print(f"  cache exists, skipped write → {out_path}")

        preview_results[paper_id] = results

        for sample_id, r in list(results.items())[:samples_per_paper]:
            print("\n" + "-" * 80)
            print(f"Sample id={sample_id}, label={r['label']}")
            print(f"Review: {r['text'][:500]}")
            display_evidence = sorted(r["evidence"], key=lambda x: x.get("score", 0), reverse=True)
            print(f"Top {len(display_evidence)} evidence (score-sorted; same priority as prompt formatting):")
            for j, ev in enumerate(display_evidence, 1):
                snippet = ev["text"][:700].replace("\n", " ")
                print(f"  #{j} score={ev['score']:.3f} section={ev['section']} "
                      f"chunk_id={ev['chunk_id']} len={len(ev['text']):,}")
                print(f"     {snippet}...")

        del retr
        torch.cuda.empty_cache(); gc.collect()

    return preview_results

# 預設先跑 dev 3 篇。若想指定 paper，可傳 paper_ids=["paper_10", "paper_102", ...]
preview_results = retrieve_preview_papers(split="dev", n_papers=3, samples_per_paper=2,
                                          force_rebuild=True, write_cache=True)


### 2.4 跑 retrieval cache（train + dev + test）
**最耗時的 cell**（約 1.5–2.5 小時 on A100）。建議跑完先 cache 重啟 session 再訓練。

每篇 paper 建一次 index，對該 paper 對應的所有 (review, paper) 做 retrieval。


In [ ]:
# Parsed sections changed, so retrieval cache must be rebuilt once too.
# Set False after the new retrieval cache is synced to Drive.
REBUILD_RETRIEVAL_CACHE = True

def retrieve_for_split(split: str, force_rebuild: bool = False):
    out_dir = Path(f"{RETRIEVED_CACHE_DIR}/{split}")
    out_dir.mkdir(parents=True, exist_ok=True)
    grouped = get_paper_ids(split)

    n_done, n_skipped = 0, 0
    for paper_id, samples in tqdm(grouped.items(), desc=f"Retrieve {split}"):
        out_path = out_dir / f"{paper_id}.json"
        if out_path.exists() and not force_rebuild:
            n_skipped += len(samples); continue

        parsed = load_parsed(paper_id)
        if parsed is None:
            print(f"  ⚠ no parsed: {paper_id}"); continue
        chunks = build_chunks(parsed)
        if not chunks:
            print(f"  ⚠ no chunks: {paper_id}"); continue

        retr = PaperRetriever(chunks)
        results = {}
        for s in samples:
            evidence = retr.retrieve(s["text"], top_k=EVIDENCE_TOP_K)
            results[s["id"]] = {"text": s["text"], "label": s["label"], "evidence": evidence}
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False)
        n_done += len(samples)
        # Free GPU memory periodically
        if n_done % 500 == 0:
            torch.cuda.empty_cache(); gc.collect()

    print(f"{split}: {n_done} retrieved, {n_skipped} skipped (cached)")

# 跑三個 split
for split in ["train", "dev", "test"]:
    retrieve_for_split(split, force_rebuild=REBUILD_RETRIEVAL_CACHE)


### 2.4.5 Sync retrieval cache → Drive（v4 fix issue D）
跑完 §2.4 後執行：把本機 `LOCAL_CACHE_DIR/retrieved` 整包 sync 到 Drive 持久化。


In [ ]:
import time
src = f"{LOCAL_CACHE_DIR}/retrieved"
dst = f"{DRIVE_CACHE_DIR}/retrieved"
total_local = sum(len(os.listdir(f'{src}/{sp}')) for sp in ['train', 'dev', 'test']
                   if os.path.exists(f'{src}/{sp}'))
print(f"Local retrieval cache: {total_local} files across train/dev/test")
if total_local == 0:
    print("⚠️  本機沒有 retrieval cache，跳過 sync")
else:
    os.makedirs(dst, exist_ok=True)
    t0 = time.time()
    rc = os.system(f'cp -r "{src}/." "{dst}/"')
    elapsed = time.time() - t0
    total_drive = sum(len(os.listdir(f'{dst}/{sp}')) for sp in ['train', 'dev', 'test']
                       if os.path.exists(f'{dst}/{sp}'))
    print(f"✅ Synced {total_drive} files Local→Drive in {elapsed:.1f}s")
    for sp in ['train', 'dev', 'test']:
        n = len(os.listdir(f'{dst}/{sp}')) if os.path.exists(f'{dst}/{sp}') else 0
        print(f"   {sp}: {n} files")


### 2.5 [QC] Recall@5 audit
手標 30 筆 dev 的 gold evidence chunk index，算 recall。**目標 recall@5 > 0.7**。

如果 < 0.7，回去跑 §2.A.3 (e5-mistral) 並 force_rebuild=True。


In [ ]:
# 抽 30 筆 dev 樣本，目視對應 paper 的最相關 chunk，判斷是否在 top 5
SAMPLE_N = 30

dev_grouped = get_paper_ids("dev")
sample_paper_ids = random.sample(list(dev_grouped.keys()), 5)
for pid in sample_paper_ids[:3]:
    with open(f"{RETRIEVED_CACHE_DIR}/dev/{pid}.json") as f:
        results = json.load(f)
    sample_id = list(results.keys())[0]
    r = results[sample_id]
    print(f"\n{'=' * 70}\nPaper: {pid}, Sample id: {sample_id}, Label: {r['label']}")
    print(f"Review: {r['text'][:300]}\n")
    evidence_sorted = sorted(r['evidence'], key=lambda x: x.get("score", 0), reverse=True)
    print(f"Top {len(evidence_sorted)} retrieved evidence (score-sorted):")
    for j, ev in enumerate(evidence_sorted, 1):
        print(f"\n  #{j} score={ev['score']:.3f} section={ev['section']}")
        print(f"     {ev['text'][:300]}...")


### 2.6 [VIZ] Evidence stats


In [ ]:
# Evidence 長度與 reranker 分數分布
all_scores = []
all_lens = []
for pid in tqdm(list(dev_grouped.keys())[:50], desc="Sampling"):
    with open(f"{RETRIEVED_CACHE_DIR}/dev/{pid}.json") as f:
        results = json.load(f)
    for s in results.values():
        for ev in s["evidence"]:
            all_scores.append(ev["score"])
            all_lens.append(len(ev["text"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(all_scores, bins=40)
axes[0].set_title("Reranker scores"); axes[0].set_xlabel("score")
axes[1].hist(all_lens, bins=40)
axes[1].set_title("Evidence length (chars)"); axes[1].set_xlabel("chars")
plt.tight_layout(); plt.show()

print(f"Reranker score: median={np.median(all_scores):.2f}, p25={np.percentile(all_scores, 25):.2f}")
print(f"Evidence len: median={int(np.median(all_lens))}, p95={int(np.percentile(all_lens, 95))}")


## §3. Prompt Templates


### 3.0 共用：載入 evidence helper + class definitions


In [ ]:
def load_retrieved(split: str, paper_id: str, sample_id: int) -> dict:
    with open(f"{RETRIEVED_CACHE_DIR}/{split}/{paper_id}.json") as f:
        results = json.load(f)
    return results[str(sample_id)]

REFERENCE_QUERY_PAT = re.compile(
    r"\b(references?|citation|citations|cite|cited|citing|referenced|bibliograph|uncited)\b",
    re.I,
)
CHECKLIST_QUERY_PAT = re.compile(
    r"\b(checklist|irb|ethics?|code of ethics|broader impacts?|institutional review board)\b",
    re.I,
)

def _looks_like_reference_evidence(ev: dict) -> bool:
    text = str(ev.get("text", ""))
    head = text[:1400].lower()
    if re.match(r"^\s*(references|bibliography)\b", head):
        return True
    if re.match(r"^\s*\[\d+\]\s+", text):
        return True
    bracket_refs = len(re.findall(r"\[\d+\]", head))
    years = len(re.findall(r"\b(?:19|20)\d{2}\b", head))
    ref_terms = sum(term in head for term in [
        "arxiv", "doi", "proceedings", "journal", "conference",
        "pmlr", "neurips", "icml", "iclr", "cvpr", "url http",
    ])
    return (bracket_refs >= 3) or (years >= 4 and ref_terms >= 1)

def _looks_like_checklist_evidence(ev: dict) -> bool:
    text = str(ev.get("text", ""))
    low = text[:2500].lower()
    if "neurips paper checklist" in low or "paper checklist" in low:
        return True
    checklist_terms = [
        "question: does the paper",
        "answer: [",
        "justification:",
        "guidelines:",
        "experimental result reproducibility",
        "experiments compute resources",
        "open access to data and code",
        "code of ethics question",
        "broader impacts question",
        "institutional review board",
    ]
    hits = sum(term in low for term in checklist_terms)
    return hits >= 3

def _filter_admin_evidence(evidence: list[dict], review_text: str = "") -> list[dict]:
    allow_refs = bool(REFERENCE_QUERY_PAT.search(review_text or ""))
    allow_checklist = bool(CHECKLIST_QUERY_PAT.search(review_text or ""))
    filtered = []
    removed_refs = removed_checklist = 0
    for ev in evidence:
        if not allow_checklist and _looks_like_checklist_evidence(ev):
            removed_checklist += 1
            continue
        if not allow_refs and _looks_like_reference_evidence(ev):
            removed_refs += 1
            continue
        filtered.append(ev)
    if (removed_refs or removed_checklist) and not filtered:
        # Avoid returning an empty context if the cached topK was all admin material.
        # This is still safer than training on an empty prompt; full retrieval rebuild can fix it later.
        return evidence[:1]
    return filtered

def format_evidence(evidence: list[dict], max_chars: int = MAX_EVIDENCE_CHARS,
                    review_text: str = "") -> str:
    # Phase 0.2 fix: sort by reranker score desc so truncation drops lowest-score chunks.
    # Big-chunk safe: if the top chunk alone exceeds max_chars, keep a truncated top chunk
    # instead of returning empty evidence.
    ev_filtered = _filter_admin_evidence(evidence, review_text=review_text)
    ev_sorted = sorted(ev_filtered, key=lambda x: x.get("score", 0), reverse=True)
    parts, total = [], 0
    for ev in ev_sorted:
        chunk = f"[{ev['section']}] {ev['text']}"
        remaining = max_chars - total
        if remaining <= 0:
            break
        if len(chunk) > remaining:
            if remaining >= 800:
                parts.append(chunk[:remaining].rstrip())
            break
        parts.append(chunk)
        total += len(chunk)
    return "\n\n---\n\n".join(parts)

# Class definitions for prompts
CLASS_DEFS = "\n".join(f"- {c['concept']}: {c['concept_desc']}" for c in CLASSES)
CONCEPT_NAMES = [c['concept'] for c in CLASSES]
print("Concepts:", CONCEPT_NAMES)


### 3.C CoT Structured Output (Variant P3)
要求模型先 quote evidence 再 output label。強迫 grounding。
**風險**：小資料下推理會幻覺。建議在 P0/P1/P2 跑通後再嘗試。


In [ ]:
SYSTEM_PROMPT_COT_LITE = f'''You are an expert reviewer evaluating academic peer review sentences for hallucinations.

Compare the review sentence against the paper context. Internally answer these in order, then choose ONE label:

Q1. Does the review explicitly cite or refer to a specific source ([N], "the paper by X", "in [Smith 2020]") AND make a claim about what that source says, proves, or does?
   -> If YES and the cited source's content does not actually say what the review claims -> Attribution Failure
   -> This rule wins over Entity when the review is about a citation's content, not its existence.

Q2. Does the review name a specific entity (model, dataset, method, metric, person, paper title, agent) that does not appear in the evidence, or appears with a different name?
   -> If YES and Q1 did not fire -> Entity

Q3. Does the review extend a specific result/finding into a universal, all-cases, always, fully-proven, or future-general claim?
   -> If YES -> Overgeneralization

Q4. Does the review state a SPECIFIC numeric value AND that exact value is wrong vs the evidence?
   -> "achieves 92% accuracy" but evidence shows 85% -> Number
   -> Just mentioning a number ("speedup of 2.2x", "with 98.7% accuracy") without the value being challenged -> NOT Number; recheck Q1/Q2/Q3
   -> Words like "numerically" or "second place" alone -> NOT Number

Q5. Is the wrongness about TIME, DATE, TENSE, or MODALITY?
   -> Year/date that disagrees with evidence (e.g., "in 2009" when paper is 2018) -> Temporal
   -> Modal verbs implying wrong certainty about future/past ("must perform", "should outperform", "will achieve", "had been") -> Temporal
   -> NOT just any past-tense verb; only when the timing/modality is the WRONG part

Categories:
{CLASS_DEFS}

Tie-breaker (when two rules fire, choose by WHAT PART is the wrong content):
- timing / date / tense / modality wrong -> Temporal
- exact numeric value wrong -> Number
- named object wrong / missing / swapped -> Entity
- claim about a cited source's content unsupported -> Attribution Failure
- local result extended too broadly -> Overgeneralization

Output ONLY the category name exactly as written. No explanation. No quotes.'''

print(f"SYSTEM_PROMPT_COT_LITE ready ({len(SYSTEM_PROMPT_COT_LITE)} chars)")


### 3.D ★ v9 Prompt Dispatcher (依 §0.6 PROMPT_VARIANT 自動選)
4 種 variant：generic / explicit_soft / few_shot / cot。


In [ ]:


def build_prompt_cot_lite(review: str, evidence_text: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT_COT_LITE},
        {"role": "user", "content": USER_TEMPLATE_0SHOT.format(evidence=evidence_text, review=review)},
    ]


# Dispatch
PROMPT_BUILDERS = {
    "cot_lite":      (build_prompt_cot_lite,      SYSTEM_PROMPT_COT_LITE),
}
if PROMPT_VARIANT not in PROMPT_BUILDERS:
    raise ValueError(f"PROMPT_VARIANT={PROMPT_VARIANT} not in {list(PROMPT_BUILDERS)}")
build_prompt, SYSTEM_PROMPT_0SHOT = PROMPT_BUILDERS[PROMPT_VARIANT]
print(f"✅ Active prompt: PROMPT_VARIANT={PROMPT_VARIANT}, builder={build_prompt.__name__}")

# Sanity check
sample = load_retrieved("dev", dev_df.iloc[0]['paper_id'], dev_df.iloc[0]['id'])
ev_text = format_evidence(sample['evidence'], review_text=sample['text'])
msgs = build_prompt(sample['text'], ev_text)


## §4. Class Imbalance Handling (Augmentation)


### 4.0 共用：Augmentation 規則


In [ ]:
NUMBER_PATTERN = re.compile(
    r"\b\d+(?:\.\d+)?(?:\s*%)?(?:\s*(?:ms|sec|min|hour|kB|MB|GB|years?|x))?\b"
)

def number_perturb(sentence: str, rng: random.Random) -> Optional[str]:
    matches = list(NUMBER_PATTERN.finditer(sentence))
    if not matches: return None
    m = rng.choice(matches)
    original = m.group(0)
    nm = re.match(r"(\d+(?:\.\d+)?)(.*)", original)
    if not nm: return None
    num_str, suffix = nm.groups(); suffix = suffix.strip()
    try: num = float(num_str)
    except: return None
    is_int = num.is_integer()
    is_year = is_int and 1900 < num < 2100 and not suffix
    is_percent = "%" in suffix
    is_decimal = not is_int and 0 < num < 1
    if is_year:
        new = int(num) + rng.choice([-5, -3, -2, -1, 1, 2, 3, 5])
    elif is_percent:
        new = max(0, min(100, num + rng.choice([-25, -15, -10, -5, 5, 10, 15, 25])))
    elif is_decimal:
        new = max(0.0, min(1.0, num + rng.choice([-0.3, -0.2, -0.1, 0.1, 0.2, 0.3])))
    elif is_int and num >= 4:
        new = int(num * rng.choice([0.5, 2.0, 0.25, 4.0]))
    else:
        new = num + rng.choice([1, 2, 3])
    if is_int:
        new_str = str(int(new))
    else:
        decimals = len(num_str.split(".")[-1]) if "." in num_str else 2
        new_str = f"{new:.{decimals}f}"
    new_token = (new_str + suffix) if suffix.startswith("%") or not suffix else f"{new_str} {suffix}"
    return sentence[:m.start()] + new_token + sentence[m.end():] if new_token != original else None

TEMPORAL_FLIPS = [
    (r"\bwill\s+", "might "), (r"\bwill\s+", "would "), (r"\bwill\s+", "could "),
    (r"\bcan\s+", "could "), (r"\bis\s+", "was "), (r"\bare\s+", "were "),
    (r"\bhas\s+", "had "), (r"\bhave\s+", "had "),
    (r"\bshows\b", "showed"), (r"\bshow\b", "showed"),
    (r"\bproposes\b", "proposed"), (r"\bachieves\b", "achieved"),
    (r"\boutperforms\b", "outperformed"), (r"\bdemonstrates\b", "demonstrated"),
    (r"\bmight\s+", "will "), (r"\bwould\s+", "will "),
    (r"\bbecomes\b", "became"), (r"\bproduces\b", "produced"),
    (r"\bimproves\b", "improved"), (r"\bgives\b", "gave"),
]

def temporal_flip(sentence: str, rng: random.Random) -> Optional[str]:
    cands = [(p, r) for p, r in TEMPORAL_FLIPS if re.search(p, sentence, re.IGNORECASE)]
    if not cands: return None
    p, r = rng.choice(cands)
    return re.sub(p, r, sentence, count=1, flags=re.IGNORECASE)

print("Augmentation functions ready.")


### 4.F ★ v9 Aug Dispatcher (依 §0.6 AUG_VARIANT 自動選)
- `T3` (default) — 純原始 train，no augmentation（v8 winner）
- `I1` — LLM paraphrase，呼叫 §4.E 生成或讀 cache
- `mechanical` (legacy fallback) — 跑 §4.A `gen_mechanical_aug()`


In [ ]:
if AUG_VARIANT == "T3":
    augmented_train_df = train_df.copy()
    if 'source_id' not in augmented_train_df.columns:
        augmented_train_df['source_id'] = pd.NA
    aug_df = pd.DataFrame(columns=['id', 'paper_id', 'source_id', 'text', 'label'])
    print(f"AUG=T3 (純原始): {len(augmented_train_df):,} train samples (no augmentation)")


else:
    raise ValueError(f"Unknown AUG_VARIANT={AUG_VARIANT}")

print("\nLabel distribution after augmentation:")
print(augmented_train_df['label'].value_counts().sort_index())


### 4.1 [VIZ] Augmentation 前後分布


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, name in [(axes[0], train_df, "Original"), (axes[1], augmented_train_df, f"After Aug ({AUG_VARIANT})")]:
    counts = df['label'].value_counts().sort_index()
    labels = [f"{i}: {ID2CONCEPT[i]}" for i in counts.index]
    sns.barplot(x=labels, y=counts.values, ax=ax, palette="viridis")
    ax.set_title(f"{name} — total {len(df):,}, max:min = {counts.max() / counts.min():.1f}:1")
    ax.tick_params(axis='x', rotation=30)
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()


## §5. QLoRA Training


### 5.1 載入 model + tokenizer
A100 走 Unsloth 路徑（bf16 + FA2）；V100 走 transformers+peft fallback (fp16)。


In [ ]:
# v8: 支援 RESUME_FROM_CHECKPOINT — 載入既有 adapter（warm start）vs fresh LoRA init

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    if RESUME_FROM_CHECKPOINT:
        # warm start：直接從 adapter checkpoint 載（含 base + LoRA）
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=RESUME_FROM_CHECKPOINT,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=DTYPE,
            load_in_4bit=True,
        )
        # 確認是 trainable 狀態
        FastLanguageModel.for_training(model)
        print(f"✅ Resumed adapter from {RESUME_FROM_CHECKPOINT}")
    else:
        # fresh LoRA init
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=DTYPE,
            load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model, r=LORA_R, lora_alpha=LORA_ALPHA,
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=SEED,
        )
        print("✅ Fresh LoRA init from base")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    )
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    if RESUME_FROM_CHECKPOINT:
        model = PeftModel.from_pretrained(base, RESUME_FROM_CHECKPOINT, is_trainable=True)
        print(f"✅ Resumed adapter from {RESUME_FROM_CHECKPOINT}")
    else:
        model = get_peft_model(base, LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA,
            target_modules=LORA_TARGET_MODULES,
            lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
        ))
        print("✅ Fresh LoRA init from base")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.truncation_side = "left"
tokenizer.padding_side = "right"
print(f"Tokenizer: truncation_side={tokenizer.truncation_side}, padding_side={tokenizer.padding_side}")


model.print_trainable_parameters() if hasattr(model, "print_trainable_parameters") else print("Model loaded")


### 5.2 建構訓練資料集
拼接 (system + user prompt + assistant=label) 到 chat template，只在 label 字串部分算 loss。


In [ ]:
def build_training_text(row: dict, evidence_text: str) -> dict:
    msgs = build_prompt(row['text'], evidence_text)
    label_text = ID2CONCEPT[row['label']]
    msgs.append({"role": "assistant", "content": label_text})
    full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": full, "label_text": label_text}

def get_evidence_for_row(split: str, paper_id: str, sample_id: int,
                          source_id=None, review_text: str = "") -> str:
    '''v3 fix issue A: augmented samples (negative id) 用 source_id 拉原始 row 的 evidence。
    原 evidence 含舊數字/時態，跟 perturb 後的 review 形成 disagreement → hallucination signal。'''
    # 決定 lookup id：augmented (sample_id < 0) 且 source_id 存在 → 用 source_id；否則用原 sample_id
    lookup_id = sample_id
    if sample_id is not None and int(sample_id) < 0 and source_id is not None and pd.notna(source_id):
        lookup_id = int(source_id)
    try:
        rec = load_retrieved(split, paper_id, lookup_id)
        return format_evidence(rec['evidence'], review_text=review_text)
    except (FileNotFoundError, KeyError):
        # Fallback: 從同 paper 的 train 任一筆借 evidence（罕見：source_id 也找不到）
        cache_path = Path(f"{RETRIEVED_CACHE_DIR}/{split}/{paper_id}.json")
        if cache_path.exists():
            with open(cache_path) as f: results = json.load(f)
            if results:
                first_key = list(results.keys())[0]
                return format_evidence(results[first_key]['evidence'], review_text=review_text)
        return "[no evidence available]"

# 對 augmented_train 建構訓練文本（傳 source_id 給 augmented samples）
print(f"Building training texts for {len(augmented_train_df):,} samples...")
train_texts, train_label_texts, train_label_ids = [], [], []
n_aug_used_source = 0
for row in tqdm(augmented_train_df.to_dict("records"), desc="Build train"):
    sid = row.get('source_id')
    ev_text = get_evidence_for_row("train", row['paper_id'], row['id'], source_id=sid,
                                   review_text=row['text'])
    if int(row['id']) < 0 and sid is not None and pd.notna(sid):
        n_aug_used_source += 1
    item = build_training_text(row, ev_text)
    train_texts.append(item['text'])
    train_label_texts.append(item['label_text'])
    train_label_ids.append(int(row['label']))  # v9: 給 V9SFTTrainer._get_train_sampler 用
print(f"Augmented samples that successfully used source_id evidence: {n_aug_used_source}")

# Dev set (small subset for eval during training, full eval in §6)
DEV_EVAL_SIZE = 300  # for eval_steps speedup
dev_eval_df = dev_df.sample(DEV_EVAL_SIZE, random_state=SEED)
dev_texts, dev_label_texts = [], []
for row in tqdm(dev_eval_df.to_dict("records"), desc="Build dev"):
    ev_text = get_evidence_for_row("dev", row['paper_id'], row['id'], review_text=row['text'])
    item = build_training_text(row, ev_text)
    dev_texts.append(item['text'])
    dev_label_texts.append(item['label_text'])

train_dataset = Dataset.from_dict({"text": train_texts})
eval_dataset = Dataset.from_dict({"text": dev_texts})
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
print(f"Sample:\n{train_texts[0]}...")


### 5.A Class weight (eff-num) + LDAM margins（v9 variant-driven）

由 §0.6 `LOSS_VARIANT` 決定：
- `ce_w` (T3 winner): effective-number-of-samples weighting (Cui et al. 2019, β=0.999)
- `ldam`: LDAM margins (Cao et al. 2019) — class-specific margin `m_y = C / n_y^(1/4)`


In [ ]:
import numpy as np

label_counts = augmented_train_df['label'].value_counts().sort_index().values

# Effective-num class weights (給 WeightedRandomSampler 用，永遠算)
effective_num = (1.0 - np.power(EFF_NUM_BETA, label_counts)) / (1.0 - EFF_NUM_BETA)
CLASS_WEIGHTS_NP = (1.0 / effective_num)
CLASS_WEIGHTS_NP = CLASS_WEIGHTS_NP / CLASS_WEIGHTS_NP.mean()

print(f"Effective-num class weights (β={EFF_NUM_BETA}):")
for cid, w in enumerate(CLASS_WEIGHTS_NP):
    print(f"  class {cid} ({CONCEPT_NAMES[cid]:22s}): count={label_counts[cid]:5d}, weight={w:.3f}")

# LDAM margins (only used if LOSS_TYPE == "ldam")
LDAM_MARGINS = LDAM_MARGIN_C / np.power(label_counts.astype(np.float32), 0.25)
LDAM_MARGINS = LDAM_MARGINS / LDAM_MARGINS.max() * 0.5  # normalize so max margin = 0.5
print(f"\nLDAM margins (C={LDAM_MARGIN_C}, scale={LDAM_SCALE}):")
for cid, m in enumerate(LDAM_MARGINS):
    print(f"  class {cid} ({CONCEPT_NAMES[cid]:22s}): margin={m:.4f}")
print(f"\nLOSS_TYPE = {LOSS_TYPE}, USE_SAMPLER = {USE_SAMPLER}")
print(f"train_label_ids ready: {len(train_label_ids)} samples")


### 5.3 Trainer 設定（v9: V9SFTTrainer + 本地 ckpt + EarlyStop always）

**v9 changes vs v8**：
- V8SFTTrainer → **V9SFTTrainer**：`compute_loss` dispatch on `LOSS_TYPE` (`"weighted_ce"` / `"ldam"`)
- focal loss 移除（v8 T5 證明不如 CE+sampler）
- `output_dir = LOCAL_CKPT_DIR/...`（**本地 SSD**，訓練期間每 save_steps 寫本地）
- `save_total_limit=2`（避免本地 SSD 爆）
- `load_best_model_at_end=True` + `EarlyStoppingCallback` **always on**（不再分 smoke/full）
- 訓練結束 §5.4.5 才把 best ckpt sync 到 Drive

**Fix Issue E (collator)**：`response_template` 用 token-id list。
**Fix Issue F (max_length)**：用 `SFTConfig`，default 1024 會截 answer span → eval=NaN。


In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from transformers import EarlyStoppingCallback
import torch.nn as nn


# ─── Fix Issue E: 抓 chat template apply 後實際 assistant header token ids ───
_dummy = [{"role": "user", "content": "x"}]
_user_only_ids = tokenizer.apply_chat_template(_dummy, tokenize=True, add_generation_prompt=True)
RESPONSE_TEMPLATE_IDS = _user_only_ids[-3:]
print(f"Response template ids = {RESPONSE_TEMPLATE_IDS} → {repr(tokenizer.decode(RESPONSE_TEMPLATE_IDS))}")

collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE_IDS,
    tokenizer=tokenizer
)

# ─── v9: V9SFTTrainer — CE/LDAM dispatch + WeightedRandomSampler ───
class V9SFTTrainer(SFTTrainer):
    '''v9: dispatch loss in {"weighted_ce", "ldam"}; sampler optional.

    Note: per-sample class-weighted CE not implemented at compute_loss level
    (DataCollatorForCompletionOnlyLM doesn't carry class_label into compute_loss).
    Instead WeightedRandomSampler does the equivalent oversampling.
    '''
    def __init__(self, *args, loss_type="weighted_ce",
                 train_label_ids=None, use_sampler=False, class_weights=None,
                 ldam_margins=None, ldam_scale=30.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_type = loss_type
        self.train_label_ids = train_label_ids
        self.use_sampler = use_sampler
        self.class_weights_np = class_weights
        self.ldam_margins_np = ldam_margins
        self.ldam_scale = ldam_scale

    def _get_train_sampler(self, *args, **kwargs):
        # 新版 transformers 簽章為 _get_train_sampler(self, dataset)
        if self.use_sampler and self.train_label_ids is not None:
            label_arr = np.array(self.train_label_ids)
            cw = self.class_weights_np if self.class_weights_np is not None else np.ones(len(CLASSES))
            sample_weights = cw[label_arr]
            print(f"[Sampler] WeightedRandomSampler — N={len(sample_weights)}, "
                  f"weight stats: min={sample_weights.min():.3f}, max={sample_weights.max():.3f}")
            return WeightedRandomSampler(weights=torch.from_numpy(sample_weights).double(),
                                          num_samples=len(label_arr),
                                          replacement=True)
        return super()._get_train_sampler(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.loss_type == "weighted_ce":
            # 等同 v8 T3 baseline：sampler 已做等效 reweight，這裡直接走 SFTTrainer 預設 CE
            return super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
        elif self.loss_type == "ldam":
            return self._ldam_loss(model, inputs, return_outputs)
        else:
            raise ValueError(f"Unknown loss_type={self.loss_type}")

    def _ldam_loss(self, model, inputs, return_outputs):
        '''LDAM (Cao et al. 2019): 在 label-token positions 對 gold class logit 減 margin，
        然後再走 CE。本任務 label 是 1 token classname (Attribution / Entity / Number / Overgeneralization / Temporal)，
        所以對所有 unmasked label tokens 用對應 class margin。'''
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  # (B, L, V)
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()  # (B, L-1)

        # margins per token：根據 gold token ID 對應到 class margin
        # 簡化：我們對每個 unmasked position，把 logit_y 減 margin_class_for_that_token
        # 因為 5 個 class name 有不同 token id，需要 build CLASS_TOKEN_IDS map
        flat_logits = shift_logits.view(-1, shift_logits.size(-1)).clone()
        flat_labels = shift_labels.view(-1)
        valid_mask = (flat_labels != -100)

        # Build class_token_id → class_index map（cache 在 self）
        if not hasattr(self, '_class_token_to_class_idx'):
            tid2cls = {}
            for cls in CLASSES:
                # take first token of class name (Attribution / Entity / Number / Overgeneralization / Temporal)
                tids = self.tokenizer(" " + cls['concept'], add_special_tokens=False)['input_ids']
                if tids:
                    tid2cls[tids[0]] = cls['id']
                tids2 = self.tokenizer(cls['concept'], add_special_tokens=False)['input_ids']
                if tids2:
                    tid2cls[tids2[0]] = cls['id']
            self._class_token_to_class_idx = tid2cls
            print(f"[LDAM] class_token_to_class_idx map ({len(tid2cls)} entries): {tid2cls}")

        margins_t = torch.tensor(self.ldam_margins_np, dtype=flat_logits.dtype, device=flat_logits.device)

        # 對 valid positions：找 gold token id → 查 class idx → 減 margin
        for tok_id, cls_idx in self._class_token_to_class_idx.items():
            position_mask = valid_mask & (flat_labels == tok_id)
            if position_mask.any():
                flat_logits[position_mask, tok_id] -= margins_t[cls_idx]

        # 套 scale + CE
        ce = nn.functional.cross_entropy(flat_logits * self.ldam_scale, flat_labels,
                                          reduction='mean', ignore_index=-100)
        return (ce, outputs) if return_outputs else ce


# ─── EXP_NAME / output_dir = 本地 SSD（v9 ckpt 策略） ───
import time
EXP_NAME = f"v9_R{RETRIEVAL_VARIANT}_H{LORA_VARIANT}_A{AUG_VARIANT}_L{LOSS_VARIANT}_P{PROMPT_VARIANT}_S{SEED}_{int(time.time())}"
output_dir = f"{LOCAL_CKPT_DIR}/{EXP_NAME}"
os.makedirs(output_dir, exist_ok=True)
print(f"📂 Local output_dir: {output_dir}")

# ─── SFTConfig：always save → load best → EarlyStop ───
training_args = SFTConfig(
    output_dir=output_dir,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=FP16, bf16=BF16,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,             # v9: 本地最多保留 2 個 checkpoint，避免 SSD 爆
    load_best_model_at_end=True,    # v9: 訓練結束自動 load best
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

EARLY_STOPPING_PATIENCE = 3
callbacks = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)]
print(f"[EarlyStop] enabled, patience={EARLY_STOPPING_PATIENCE} on metric={training_args.metric_for_best_model}")

trainer = V9SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    data_collator=collator,
    # callbacks=callbacks,
    loss_type=LOSS_TYPE,
    train_label_ids=train_label_ids,
    use_sampler=USE_SAMPLER,
    class_weights=CLASS_WEIGHTS_NP,
    ldam_margins=LDAM_MARGINS if LOSS_TYPE == "ldam" else None,
    ldam_scale=LDAM_SCALE,
)



### 5.4 訓練

**前置條件**：§5.3.5 必須 PASS（length OK + mask OK），否則必 NaN。

訓練中觀察：
- step 200 train_loss 應 < 2.5（base Qwen 對短 classification continuation 的 ballpark）
- step 200 val_loss 應為**有限正數**（不是 NaN）
- 若 step 400 仍 NaN 或 loss 暴衝，立刻 stop 並回 §5.3.5 重新診斷


In [ ]:
from torch.utils.data import WeightedRandomSampler


print(f"Output dir: {output_dir}")
print("Starting training...")
trainer_stats = trainer.train()
trainer.save_model(output_dir)
print(f"\n✅ Training done. Saved to {output_dir}")
print(f"  best_model_checkpoint = {trainer.state.best_model_checkpoint}")
print(f"  best_metric           = {trainer.state.best_metric}")


### 5.4.5 ★ NEW v9 — Sync best checkpoint to Drive
訓練結束後執行：把 `trainer.state.best_model_checkpoint`（已被 EarlyStop+load_best 選出）
從本地 SSD 複製到 Drive，刪本地其他 checkpoint 釋放 SSD（保留 best 一份本地以利 §6 inference）。


In [ ]:
import shutil

best_ckpt_local = trainer.state.best_model_checkpoint
if not best_ckpt_local or not os.path.exists(best_ckpt_local):
    # fallback: 用 trainer.save_model() 寫的 output_dir 本身
    best_ckpt_local = output_dir
    print(f"⚠ best_model_checkpoint 為空或不存在，fallback 用 output_dir = {best_ckpt_local}")

drive_target = f"{DRIVE_CKPT_DIR}/{EXP_NAME}_best"
print(f"Sync best ckpt:")
print(f"  src: {best_ckpt_local}")
print(f"  dst: {drive_target}")
if os.path.exists(drive_target):
    print(f"  ⚠ Drive target already exists, removing first...")
    shutil.rmtree(drive_target)
shutil.copytree(best_ckpt_local, drive_target)

# 統計大小
total_mb = sum(os.path.getsize(os.path.join(dp, f))
                for dp, _, fs in os.walk(drive_target) for f in fs) / 1024 / 1024
print(f"✅ Synced ({total_mb:.1f} MB)")
print(f"   eval at this ckpt: {trainer.state.best_metric}")

# 刪本地其他 checkpoint，保留 best 一份本地（給 §6 inference 用）
parent = Path(output_dir)
removed = 0
for ck in parent.glob("checkpoint-*"):
    if str(ck) != best_ckpt_local:
        shutil.rmtree(ck)
        removed += 1
print(f"🗑 Removed {removed} non-best local checkpoint(s) (kept {best_ckpt_local} for inference)")

# 寫一個 marker 紀錄當前 best ckpt path（給 §6 / §7 用）
LATEST_BEST_CKPT = best_ckpt_local
LATEST_BEST_CKPT_DRIVE = drive_target
print(f"\nLATEST_BEST_CKPT (local for inference) = {LATEST_BEST_CKPT}")
print(f"LATEST_BEST_CKPT_DRIVE                 = {LATEST_BEST_CKPT_DRIVE}")


### 5.5 [VIZ] Loss curve


In [ ]:
log_history = trainer.state.log_history
train_log = [(l['step'], l['loss']) for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_log = [(l['step'], l['eval_loss']) for l in log_history if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(10, 4))
if train_log:
    s, v = zip(*train_log); ax.plot(s, v, label='train_loss', alpha=0.7)
if eval_log:
    s, v = zip(*eval_log); ax.plot(s, v, label='eval_loss', marker='o')
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("Training Curves")
plt.tight_layout(); plt.show()


## §6. Evaluation on Dev Set


### 6.1 Inference helper


In [ ]:
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)

def parse_label(output: str) -> int:
    out = output.strip().lower()
    for cls in CLASSES:
        if cls['concept'].lower() in out:
            return cls['id']
    if 'attribut' in out: return 0
    if 'entity' in out: return 1
    if 'number' in out: return 2
    if 'overgeneral' in out or 'general' in out: return 3
    if 'temporal' in out or 'tense' in out: return 4
    return 0  # default to most common

@torch.no_grad()
def predict_one(review: str, evidence_text: str, temp: float = 0.0) -> tuple[int, str]:
    '''v9: 加 temp 參數支援 TTA。temp=0.0 → greedy；> 0 → sampling。'''
    msgs = build_prompt(review, evidence_text)
    prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    do_sample = temp > 0.0
    out = model.generate(
        **inputs, max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=do_sample,
        temperature=(temp if do_sample else 1.0),
        top_p=(0.9 if do_sample else 1.0),
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = out[0, inputs.input_ids.shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_label(text), text

# Sanity check
ev_text = format_evidence(load_retrieved("dev", dev_df.iloc[0]['paper_id'], dev_df.iloc[0]['id'])['evidence'])
pred_id, pred_str = predict_one(dev_df.iloc[0]['text'], ev_text)
print(f"Sample dev prediction: pred='{pred_str}' → id={pred_id}, gold={dev_df.iloc[0]['label']}")


### 6.2 Predict 全 dev set（v9 INFERENCE_VARIANT-aware）
Greedy（默認）or TTA majority vote — 由 §0.6 `INFERENCE_VARIANT` 決定。
**TTA 推論時間 ~25 min × K**（K=3 → 75 min on full dev）。


In [ ]:
def predict_split_with_inference_variant(split_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    '''Run inference on a split with the active INFERENCE_VARIANT (greedy/tta_k3/self_consist_k5).'''
    # K passes (K=1 for greedy)
    all_passes = []  # list of [pred_id_per_sample] per temperature pass
    raw_collect = []  # store last raw string per sample
    for k_idx, t in enumerate(TTA_TEMPS):
        pass_preds = []
        pass_raws = []
        for row in tqdm(split_df.to_dict("records"), desc=f"{split_name} pass {k_idx+1}/{len(TTA_TEMPS)} (temp={t})"):
            ev_text = get_evidence_for_row(split_name, row['paper_id'], row['id'])
            pid, pstr = predict_one(row['text'], ev_text, temp=t)
            pass_preds.append(pid)
            pass_raws.append(pstr)
        all_passes.append(pass_preds)
        raw_collect = pass_raws  # keep last for record

    # Vote across passes (single pass for greedy)
    final_preds = []
    for i in range(len(split_df)):
        votes = [all_passes[k][i] for k in range(len(all_passes))]
        winner = Counter(votes).most_common(1)[0][0]
        final_preds.append(winner)

    rows = []
    for i, row in enumerate(split_df.to_dict("records")):
        rec = {"id": row['id'], "paper_id": row['paper_id'], "text": row['text'],
               "pred": final_preds[i], "pred_raw": raw_collect[i]}
        if 'label' in row:
            rec['gold'] = row['label']
        if len(all_passes) > 1:
            rec['pred_passes'] = ','.join(str(p) for p in [all_passes[k][i] for k in range(len(all_passes))])
        rows.append(rec)
    return pd.DataFrame(rows)


DEV_SAMPLE_N = 500
DEV_SUFFIX = f"_{INFERENCE_VARIANT}_seed{SEED}_dev{DEV_SAMPLE_N}"


dev_split_df = dev_df.sample(n=min(DEV_SAMPLE_N, len(dev_df)), random_state=SEED)

pred_df = predict_split_with_inference_variant(dev_split_df, "dev")

out_path = f"{OUTPUTS_DIR}/{EXP_NAME}_dev{DEV_SUFFIX}.csv"
pred_df.to_csv(out_path, index=False)
print(f"Saved {out_path}")

# Cache predictions JSON for ensemble (§6.6)
pred_cache_dir = Path(f"{CACHE_DIR}/predictions")
pred_cache_dir.mkdir(parents=True, exist_ok=True)
pred_json_path = pred_cache_dir / f"dev_seed{SEED}_R{RETRIEVAL_VARIANT}_H{LORA_VARIANT}_A{AUG_VARIANT}_L{LOSS_VARIANT}_P{PROMPT_VARIANT}_I{INFERENCE_VARIANT}.json"
json.dump([{"id": int(r['id']), "pred": int(r['pred'])} for r in pred_df.to_dict("records")],
          open(pred_json_path, "w"))
print(f"Cached predictions for ensemble → {pred_json_path}")


### 6.3 Macro F1 + Per-class Report


In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

y_true = pred_df['gold'].values
y_pred = pred_df['pred'].values

macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f"\n🎯 Dev Macro F1: {macro_f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=[c['concept'] for c in CLASSES], digits=4))


## §7. Test Set Inference & Submission


### 7.1 Predict 全 test set（v9 INFERENCE_VARIANT-aware）
重用 §6.2 的 `predict_split_with_inference_variant`，跑完 test 後 cache predictions JSON
讓 §7.2 ensemble-aware（如 multi-seed，最終 submission 從 cached test predictions vote）。


In [ ]:
test_pred_df = predict_split_with_inference_variant(test_df, "test")
test_pred_df = test_pred_df.rename(columns={"pred": "label"})  # submission 格式

# Cache test predictions JSON for §7.2 ensemble vote
pred_cache_dir = Path(f"{CACHE_DIR}/predictions")
pred_cache_dir.mkdir(parents=True, exist_ok=True)
test_json_path = pred_cache_dir / f"test_seed{SEED}_R{RETRIEVAL_VARIANT}_H{LORA_VARIANT}_A{AUG_VARIANT}_L{LOSS_VARIANT}_P{PROMPT_VARIANT}_I{INFERENCE_VARIANT}.json"
json.dump([{"id": int(r['id']), "label": int(r['label'])} for r in test_pred_df.to_dict("records")],
          open(test_json_path, "w"))
print(f"Cached test predictions → {test_json_path}")

print(f"\nPredicted {len(test_pred_df)} test samples")
print("Distribution:")
print(test_pred_df['label'].value_counts().sort_index())


### 7.2 寫 submission CSV（v9 ensemble-aware）
- 單 seed: 直接寫 §7.1 結果
- Multi-seed: 從 cached test predictions JSON 各 seed 讀回，majority vote → 寫 ensemble submission


In [ ]:
if len(ENSEMBLE_SEEDS) <= 1:
    submission = test_pred_df[['id', 'label']]
else:
    # Ensemble: read each seed's cached test predictions, vote
    from collections import Counter as _C
    pred_cache_dir = Path(f"{CACHE_DIR}/predictions")
    seed_test_files = []
    for s in ENSEMBLE_SEEDS:
        pattern = f"test_seed{s}_R{RETRIEVAL_VARIANT}_H{LORA_VARIANT}_A{AUG_VARIANT}_L{LOSS_VARIANT}_P{PROMPT_VARIANT}_*.json"
        matched = sorted(pred_cache_dir.glob(pattern))
        if matched: seed_test_files.append(matched[-1])
    if len(seed_test_files) < 2:
        print(f"⚠ Only {len(seed_test_files)} seed test files; falling back to single-seed submission.")
        submission = test_pred_df[['id', 'label']]
    else:
        seed_preds = []
        for f in seed_test_files:
            preds = json.load(open(f))
            preds_by_id = {p['id']: p['label'] for p in preds}
            seed_preds.append(preds_by_id)
            print(f"  loaded {len(preds_by_id)} test preds from {f.name}")
        rows = []
        for _, row in test_df.iterrows():
            votes = [seed_preds[k].get(int(row['id'])) for k in range(len(seed_preds))]
            votes = [v for v in votes if v is not None]
            if not votes: continue
            winner = _C(votes).most_common(1)[0][0]
            rows.append({"id": int(row['id']), "label": winner})
        submission = pd.DataFrame(rows)
        print(f"\n🗳 Ensemble submission ({len(seed_test_files)} seeds) ready: {len(submission)} rows")

out_path = f"{OUTPUTS_DIR}/predictions_{EXP_NAME}.csv"
submission.to_csv(out_path, index=False)
print(f"✅ Saved {out_path}")

# 也存一份不含 exp_name 的當 latest
latest_path = f"{OUTPUTS_DIR}/predictions.csv"
submission.to_csv(latest_path, index=False)
print(f"✅ Saved {latest_path} (latest)")

print("\n--- 前 10 筆 ---")
print(submission.head(10).to_string())


### 7.3 [QC] 預測分布合理性
不該全部 0 或全部 1。應該分布大致接近 dev 比例。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = test_pred_df['label'].value_counts().sort_index()
labels = [f"{i}: {ID2CONCEPT[i]}" for i in counts.index]
sns.barplot(x=labels, y=counts.values, ax=ax, palette="viridis")
ax.set_title(f"Test predictions distribution (total {len(test_pred_df)})")
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    ax.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()

print("\n--- 預期 vs 實際 ---")
expected_pct = train_df['label'].value_counts(normalize=True).sort_index() * 100
actual_pct = test_pred_df['label'].value_counts(normalize=True).sort_index() * 100
for i in range(5):
    e = expected_pct.get(i, 0); a = actual_pct.get(i, 0)
    print(f"  {ID2CONCEPT[i]:25s}: expected ~{e:5.1f}% / actual {a:5.1f}%")
